# CCLAP analysis

This notebook reproduces the main analyses reported for CCLAP using the compact public release files. The child association data are de-identified. 

Input files used in this notebook:

- `cclap_child_association_public.csv`: de-identified child association responses with raw responses, segmented tokens/POS, and retained cleaned tokens/POS.
- `analysis_relation_strengths.csv`: derived cue-response or word-pair strengths for child associations, adult associations, child-directed speech, and child-directed books.
- `analysis_word_measures.csv`: derived word-level measures used in the analyses, including child production frequency, speech/book frequency and contextual diversity, CCLOWW frequency, and WordBank AoA.


## 1. Clean segmented child responses

The segmented responses are filtered by vocabulary and POS tags. Repeated cue-associate pairs from the same child are counted once.

In [ ]:
from __future__ import annotations

import csv
from collections import Counter
from pathlib import Path
from typing import Iterable

INPUT_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
CLEAN_DIR = OUTPUT_DIR / 'cleaned_data'
DATA_DIR = OUTPUT_DIR / 'analysis_data'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIG_DIR = OUTPUT_DIR / 'figures'
for directory in [INPUT_DIR, OUTPUT_DIR, CLEAN_DIR, DATA_DIR, TABLE_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CHILD_INPUT = INPUT_DIR / 'cclap_child_association_public.csv'
RELATION_PATH = INPUT_DIR / 'analysis_relation_strengths.csv'
WORD_MEASURES_PATH = INPUT_DIR / 'analysis_word_measures.csv'
OUTPUT_ROWS = CLEAN_DIR / 'responses_with_pos.csv'
OUTPUT_COUNTS = CLEAN_DIR / 'response_token_counts.csv'
OUTPUT_SUMMARY = CLEAN_DIR / 'cue_response_summary.csv'
OUTPUT_SCREENED = CLEAN_DIR / 'vocab_screened_out_responses.csv'

for path in [CHILD_INPUT, RELATION_PATH, WORD_MEASURES_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print('Input folder:', INPUT_DIR)
print('Output folder:', OUTPUT_DIR)
print('Cleaned-response output:', OUTPUT_ROWS.name)
print('Cue-response count output:', OUTPUT_COUNTS.name)
print('Cue summary output:', OUTPUT_SUMMARY.name)


In [ ]:
def parse_token_string(s: str) -> list[tuple[str, str]]:
    out = []
    for item in (s or '').split():
        if '/' not in item:
            continue
        tok, pos = item.rsplit('/', 1)
        out.append((tok, pos))
    return out

CONTENT_FASTHAN_TAGS = {
    'NN', 'NR', 'NT',
    'VV',
    'JJ', 'VA',
    'AD',
    'PN',
    'CD', 'OD',
    'M',
}

def is_content_pos(pos: str) -> bool:
    return (pos or '').strip().upper() in CONTENT_FASTHAN_TAGS

CUE_CLASS_MANUAL_OVERRIDES = {
    '保护': 'Verb',
    '垃圾': 'Noun', '感觉': 'Noun', '棒棒糖': 'Noun', '母鸡': 'Noun',
    '营养': 'Noun', '运动': 'Noun', '铃铛': 'Noun', '阳光': 'Noun',
    '小心': 'Adj/Adv', '有毒': 'Adj/Adv', '棕色': 'Adj/Adv',
    '毛茸茸': 'Adj/Adv', '电动': 'Adj/Adv', '紫色': 'Adj/Adv',
}

def classify_cue_pos(cue: str, pos: str) -> tuple[str, str, bool]:
    if cue in CUE_CLASS_MANUAL_OVERRIDES:
        label = CUE_CLASS_MANUAL_OVERRIDES[cue]
        return label, label, True
    pos = (pos or '').strip().upper()
    if pos in {'NN', 'NR', 'NT', 'IJ'}:
        return 'Noun', 'Noun', False
    if pos == 'VV':
        return 'Verb', 'Verb', False
    if pos in {'VA', 'JJ', 'AD'}:
        return 'Adj/Adv', 'Adj/Adv', False
    return 'Other', 'Other', False

CUE_CLASS_EN = {'Noun': 'Noun', 'Verb': 'Verb', 'Adj/Adv': 'Adj/Adv', 'Other': 'Other'}

tokenized_rows = []
with CHILD_INPUT.open('r', encoding='utf-8-sig', newline='') as f:
    for row in csv.DictReader(f):
        tokenized_rows.append(row)

print('Child association rows:', len(tokenized_rows))
tokenized_rows[:3]


In [ ]:
rows_for_csv: list[dict[str, str]] = []
counts_for_csv: list[dict[str, object]] = []
summary_tracker: dict[str, dict[str, int | str]] = {}
seen_count_keys: set[tuple[str, str, str]] = set()

for row in tokenized_rows:
    cleaned_tokens_with_pos = row.get('cleaned_tokens_with_pos', '') or ''
    kept = parse_token_string(cleaned_tokens_with_pos)

    rows_for_csv.append({
        'subj': row['subj'],
        'cue': row['cue'],
        'response': cleaned_tokens_with_pos,
    })

    cue_summary = summary_tracker.setdefault(row['cue'], {'n_subj': 0, 'n_DK': 0, 'cue_pos': row.get('cue_pos', '')})
    cue_summary['n_subj'] += 1
    if str(row.get('is_dk', '')) == '1':
        cue_summary['n_DK'] += 1

    filtered_tokens = [(token, pos) for token, pos in kept if token != row['cue']]
    unique_tokens = {}
    for token, pos in filtered_tokens:
        unique_tokens.setdefault(token, pos)
    for token, pos in sorted(unique_tokens.items()):
        count_key = (row['subj'], row['cue'], token)
        if count_key in seen_count_keys:
            continue
        seen_count_keys.add(count_key)
        counts_for_csv.append({
            'subj': row['subj'],
            'cue': row['cue'],
            'response': token,
            'POS': pos,
            'count': 1,
        })

summary_rows = []
for cue, stats in sorted(summary_tracker.items()):
    cue_class, cue_class_corrected, cue_class_manual_corrected = classify_cue_pos(cue, str(stats['cue_pos']))
    summary_rows.append({
        'cue': cue,
        'cue_POS': stats['cue_pos'],
        'cue_class_4way': cue_class,
        'cue_class_4way_en': CUE_CLASS_EN[cue_class],
        'cue_class_4way_corrected': cue_class_corrected,
        'cue_class_4way_corrected_en': CUE_CLASS_EN[cue_class_corrected],
        'cue_class_manual_corrected': '1' if cue_class_manual_corrected else '0',
        'n_subj': stats['n_subj'],
        'n_DK': stats['n_DK'],
        'n_valid': stats['n_subj'] - stats['n_DK'],
    })

with OUTPUT_ROWS.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['subj', 'cue', 'response'])
    writer.writeheader()
    writer.writerows(rows_for_csv)

with OUTPUT_COUNTS.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['subj', 'cue', 'response', 'POS', 'count'])
    writer.writeheader()
    writer.writerows(counts_for_csv)

with OUTPUT_SUMMARY.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['cue', 'cue_POS', 'cue_class_4way', 'cue_class_4way_en', 'cue_class_4way_corrected', 'cue_class_4way_corrected_en', 'cue_class_manual_corrected', 'n_subj', 'n_DK', 'n_valid'])
    writer.writeheader()
    writer.writerows(summary_rows)

with OUTPUT_SCREENED.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['note'])
    writer.writeheader()
    writer.writerow({'note': 'Screening was applied before creating the public child association file.'})

print('responses_with_pos rows:', len(rows_for_csv))
print('response_token_counts rows:', len(counts_for_csv))
print('cue_response_summary rows:', len(summary_rows))


## 2. Descriptive summaries

This section summarizes the child association responses retained for analysis.

In [ ]:
import pandas as pd
import numpy as np

# The analysis unit is a file-level experimental record.
tok = pd.read_csv(CHILD_INPUT, encoding='utf-8-sig')
tok['subj'] = tok['subj'].astype(str).str.strip()
tok['cue'] = tok['cue'].astype(str).str.strip()
tok['source_file'] = tok['source_file'].astype(str).str.strip()
actual_subj = sorted(tok['subj'].unique())
print('Experimental records in the association task:', len(actual_subj))

if 'child_identity' not in tok.columns:
    tok['child_identity'] = tok['subj']
tok['child_identity'] = tok['child_identity'].astype(str).str.strip()
source_file_per_child = (
    tok.groupby('child_identity')
    .agg(
        n_source_files=('source_file', 'nunique'),
        n_subj_ids=('subj', 'nunique'),
        n_cues=('cue', 'nunique'),
        n_rows=('cue', 'size'),
    )
    .reset_index()
)
multi_file_children = source_file_per_child.loc[source_file_per_child['n_source_files'] > 1].copy()
print('Unique child identities, supplementary only:', source_file_per_child['child_identity'].nunique())
print('Children with multiple file-level samples, supplementary only:', len(multi_file_children))
print('Distribution of file-level samples per child, supplementary only:')
print(source_file_per_child['n_source_files'].value_counts().sort_index().to_string())

subj_demo = tok[['subj', 'age', 'gender']].drop_duplicates('subj').copy()
subj_demo['age'] = pd.to_numeric(subj_demo['age'], errors='coerce')
cue_per_subj = tok.groupby('subj')['cue'].nunique()

n_total = len(subj_demo)
gender_norm = subj_demo['gender'].astype(str).str.strip().str.lower()
n_female = int(gender_norm.isin(['female', 'f', '女']).sum())
n_male = int(gender_norm.isin(['male', 'm', '男']).sum())
age = subj_demo['age'].dropna()
q = cue_per_subj

print(f'Participant-samples: n = {n_total}')
print(f'Female = {n_female} ({n_female/n_total:.1%}), male = {n_male} ({n_male/n_total:.1%})')
print(f'Age: M = {age.mean():.2f}, SD = {age.std(ddof=1):.2f}, range = [{age.min():.2f}, {age.max():.2f}]')
print(f'Cues answered per experimental record: M = {q.mean():.1f}, SD = {q.std(ddof=1):.1f}, range = [{q.min()}, {q.max()}]')

source_file_per_child.to_csv(TABLE_DIR / 'descriptives_subject_source_files.csv', index=False, encoding='utf-8-sig')

n_child_identities = source_file_per_child['child_identity'].nunique()
n_multi_file_children = len(multi_file_children)
source_file_distribution = ', '.join(
    f'{int(k)} file-level sample(s): {int(v)}'
    for k, v in source_file_per_child['n_source_files'].value_counts().sort_index().items()
)
subject_descriptive_rows = [
    ['Participant-samples (file-level subj, N)', f'{n_total}'],
    ['Female, n (%)', f'{n_female} ({n_female/n_total:.1%})'],
    ['Male, n (%)', f'{n_male} ({n_male/n_total:.1%})'],
    ['Age in years: M (SD)', f'{age.mean():.2f} ({age.std(ddof=1):.2f})'],
    ['Age range', f'[{age.min():.2f}, {age.max():.2f}]'],
    ['Cues answered per experimental record: M (SD)', f'{q.mean():.1f} ({q.std(ddof=1):.1f})'],
    ['Cues answered per experimental record range', f'[{q.min()}, {q.max()}]'],
    ['Supplementary: unique child identities', f'{n_child_identities}'],
    ['Supplementary: children with multiple file-level samples', f'{n_multi_file_children}'],
    ['Supplementary: file-level samples per child distribution', source_file_distribution],
]
pd.DataFrame(subject_descriptive_rows, columns=['Measure', 'Value']).to_csv(
    TABLE_DIR / 'descriptives_subjects.csv', index=False, encoding='utf-8-sig')
print('Saved: descriptives_subjects.csv')


In [ ]:
# Cue-level response counts.
summary = pd.read_csv(OUTPUT_SUMMARY, encoding='utf-8-sig')
print('Number of cues:', len(summary))

n_subj = summary['n_subj']
n_valid = summary['n_valid']
print(f'Responses per cue including DK: M = {n_subj.mean():.1f}, SD = {n_subj.std(ddof=1):.1f}, range = [{n_subj.min()}, {n_subj.max()}]')
print(f'Valid responses per cue excluding DK: M = {n_valid.mean():.1f}, SD = {n_valid.std(ddof=1):.1f}, range = [{n_valid.min()}, {n_valid.max()}]')

cue_descriptive_rows = [
    ['Number of cues', f'{len(summary)}'],
    ['Responses per cue including DK: M (SD)', f'{n_subj.mean():.1f} ({n_subj.std(ddof=1):.1f})'],
    ['Responses per cue including DK range', f'[{n_subj.min()}, {n_subj.max()}]'],
    ['Valid responses per cue excluding DK: M (SD)', f'{n_valid.mean():.1f} ({n_valid.std(ddof=1):.1f})'],
    ['Valid responses per cue excluding DK range', f'[{n_valid.min()}, {n_valid.max()}]'],
]
pd.DataFrame(cue_descriptive_rows, columns=['Measure', 'Value']).to_csv(
    TABLE_DIR / 'descriptives_cues.csv', index=False, encoding='utf-8-sig')
print('Saved: descriptives_cues.csv')


In [ ]:
# ---- cue-level associate richness and response coverage ----
counts = pd.read_csv(OUTPUT_COUNTS, encoding='utf-8-sig')
counts['count'] = pd.to_numeric(counts['count'], errors='coerce').fillna(1)

cue_vocab = set(summary['cue'].astype(str))
counts['response_in_cue_vocab'] = counts['response'].astype(str).isin(cue_vocab)

cue_associate_coverage = (
    counts.groupby('cue')
    .agg(
        n_unique_associates=('response', 'nunique'),
        n_response_tokens=('count', 'sum'),
        n_response_tokens_in_cue_vocab=('response_in_cue_vocab', 'sum'),
    )
    .reset_index()
)
cue_associate_coverage['response_token_coverage_in_cue_vocab'] = (
    cue_associate_coverage['n_response_tokens_in_cue_vocab']
    / cue_associate_coverage['n_response_tokens']
)
cue_associate_coverage = summary[['cue']].merge(cue_associate_coverage, on='cue', how='left')
cue_associate_coverage[['n_unique_associates', 'n_response_tokens', 'n_response_tokens_in_cue_vocab']] = (
    cue_associate_coverage[['n_unique_associates', 'n_response_tokens', 'n_response_tokens_in_cue_vocab']]
    .fillna(0)
    .astype(int)
)

n_unique_associates = cue_associate_coverage['n_unique_associates']
coverage = cue_associate_coverage['response_token_coverage_in_cue_vocab']
token_coverage = counts['response_in_cue_vocab'].mean()
pair_coverage = counts[['cue', 'response', 'response_in_cue_vocab']].drop_duplicates()['response_in_cue_vocab'].mean()
response_type_coverage = counts[['response', 'response_in_cue_vocab']].drop_duplicates()['response_in_cue_vocab'].mean()

print(f'Unique associates per cue: M = {n_unique_associates.mean():.1f}, SD = {n_unique_associates.std(ddof=1):.1f}, range = [{n_unique_associates.min()}, {n_unique_associates.max()}]')
print(f'SWOW-style response-token coverage: {token_coverage:.3f} ({token_coverage * 100:.1f}%)')
print(f'Cue-response pair coverage: {pair_coverage:.3f} ({pair_coverage * 100:.1f}%)')
print(f'Unique response vocabulary coverage: {response_type_coverage:.3f} ({response_type_coverage * 100:.1f}%)')
print(f'Cue-wise response-token coverage: M = {coverage.mean():.3f}, SD = {coverage.std(ddof=1):.3f}, range = [{coverage.min():.3f}, {coverage.max():.3f}]')

cue_associate_coverage.to_csv(TABLE_DIR / 'cue_associate_coverage_summary.csv', index=False, encoding='utf-8-sig')

cue_descriptive_rows.extend([
    ['Unique associates per cue: M (SD)', f'{n_unique_associates.mean():.1f} ({n_unique_associates.std(ddof=1):.1f})'],
    ['Unique associates per cue range', f'[{n_unique_associates.min()}, {n_unique_associates.max()}]'],
    ['SWOW-style response-token coverage', f'{token_coverage:.3f} ({token_coverage * 100:.1f}%)'],
    ['Cue-wise response-token coverage: M (SD)', f'{coverage.mean():.3f} ({coverage.std(ddof=1):.3f})'],
    ['Cue-wise response-token coverage range', f'[{coverage.min():.3f}, {coverage.max():.3f}]'],
])
pd.DataFrame(cue_descriptive_rows, columns=['Measure', 'Value']).to_csv(
    TABLE_DIR / 'descriptives_cues.csv', index=False, encoding='utf-8-sig')
print('\nSaved: cue_associate_coverage_summary.csv')
print('Updated: descriptives_cues.csv')

In [ ]:
# ---- retained associate POS / word-class distribution ----
import matplotlib.pyplot as plt
import seaborn as sns

counts = pd.read_csv(OUTPUT_COUNTS, encoding='utf-8-sig')
counts['POS'] = counts['POS'].astype(str).str.strip()
counts['response'] = counts['response'].astype(str).str.strip()
counts['count'] = pd.to_numeric(counts['count'], errors='coerce').fillna(1)

POS_LABELS = {
    'NN': 'Noun', 'NR': 'Noun', 'NT': 'Noun',
    'VV': 'Verb', 'VC': 'Verb', 'VE': 'Verb',
    'JJ': 'Adjective', 'VA': 'Adjective',
    'AD': 'Adverb',
    'PN': 'Pronoun',
    'CD': 'Numeral', 'OD': 'Numeral',
    'M': 'Measure word',
}
POS_ORDER = ['Noun', 'Verb', 'Adjective', 'Adverb', 'Pronoun', 'Numeral', 'Measure word']
POS_COLORS = {
    'Noun': '#8FB9AA',
    'Verb': '#D9A441',
    'Adjective': '#C9866A',
    'Adverb': '#7C9EBC',
    'Pronoun': '#B9A7C9',
    'Numeral': '#A7A26D',
    'Measure word': '#C8B89A',
}

counts['word_class'] = counts['POS'].map(POS_LABELS).fillna('Other')

pos_contribution_distribution = (
    counts.groupby('word_class', as_index=False)
    .agg(n=('count', 'sum'))
)
pos_contribution_distribution['unit'] = 'Associate tokens'

# A response word can occasionally receive more than one POS tag across contexts.
# For type-level counts, assign each word to the word class with the largest retained contribution count.
response_class_counts = (
    counts.groupby(['response', 'word_class'], as_index=False)
    .agg(n=('count', 'sum'))
    .sort_values(['response', 'n', 'word_class'], ascending=[True, False, True])
)
dominant_response_class = response_class_counts.drop_duplicates('response', keep='first')
pos_type_distribution = (
    dominant_response_class.groupby('word_class', as_index=False)
    .agg(n=('response', 'nunique'))
)
pos_type_distribution['unit'] = 'Associate types'

pos_distribution = pd.concat([pos_contribution_distribution, pos_type_distribution], ignore_index=True)
pos_distribution['word_class'] = pd.Categorical(pos_distribution['word_class'], categories=POS_ORDER, ordered=True)
pos_distribution = pos_distribution.sort_values(['unit', 'word_class']).reset_index(drop=True)
pos_distribution['proportion'] = pos_distribution.groupby('unit')['n'].transform(lambda x: x / x.sum())
pos_distribution.to_csv(TABLE_DIR / 'retained_associate_pos_distribution.csv', index=False, encoding='utf-8-sig')

print('Retained associate POS distribution:')
display(pos_distribution.assign(percent=pos_distribution['proportion'] * 100).round({'proportion': 3, 'percent': 1}))

sns.set_theme(style='white', font='Arial')
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.5), sharey=True)
for ax, unit in zip(axes, ['Associate tokens', 'Associate types']):
    data = pos_distribution.loc[pos_distribution['unit'].eq(unit)].copy()
    sns.barplot(
        data=data,
        x='word_class',
        y='proportion',
        order=POS_ORDER,
        palette=[POS_COLORS[label] for label in POS_ORDER],
        ax=ax,
        edgecolor='white',
        linewidth=0.8,
    )
    for patch, value in zip(ax.patches, data['proportion']):
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            value + 0.012,
            f'{value * 100:.1f}%',
            ha='center',
            va='bottom',
            fontsize=10.8,
            color='#333333',
        )
    ax.set_title(unit, fontsize=13.5, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylim(0, max(0.55, pos_distribution['proportion'].max() + 0.08))
    ax.tick_params(axis='x', rotation=32, labelsize=10.2)
    ax.tick_params(axis='y', labelsize=11.0)
    ax.grid(False)
axes[0].set_ylabel('Proportion', fontsize=12.2)
axes[1].set_ylabel('')
sns.despine(fig=fig, left=False, bottom=False)
fig.tight_layout(w_pad=1.2)
fig.savefig(FIG_DIR / 'figures_associate_pos_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures_associate_pos_distribution.png')
print('Saved: retained_associate_pos_distribution.csv')

In [ ]:
# ---- cue-level contributor diversity ----
counts = pd.read_csv(OUTPUT_COUNTS, encoding='utf-8-sig')
counts['count'] = pd.to_numeric(counts['count'], errors='coerce').fillna(1)

# response_token_counts is already deduplicated within subj-cue-response; one row = one retained associate contribution.
subj_contrib = (
    counts.groupby(['cue', 'subj'], as_index=False)
    .agg(n_associate_contributions=('count', 'sum'))
)

def simpson_concentration_from_counts(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors='coerce').fillna(0)
    total = values.sum()
    if total <= 1:
        return np.nan
    return float((values * (values - 1)).sum() / (total * (total - 1)))

cue_contributor_diversity = (
    subj_contrib.groupby('cue')
    .agg(
        n_contributing_subj=('subj', 'nunique'),
        n_associate_contributions=('n_associate_contributions', 'sum'),
        simpson_concentration=('n_associate_contributions', simpson_concentration_from_counts),
    )
    .reset_index()
)
cue_contributor_diversity['simpson_contributor_diversity'] = 1 - cue_contributor_diversity['simpson_concentration']
cue_contributor_diversity = summary[['cue', 'n_subj', 'n_valid']].merge(
    cue_contributor_diversity, on='cue', how='left'
)

for col in ['n_contributing_subj', 'n_associate_contributions']:
    cue_contributor_diversity[col] = cue_contributor_diversity[col].fillna(0).astype(int)

simpson_conc = cue_contributor_diversity['simpson_concentration']
simpson_div = cue_contributor_diversity['simpson_contributor_diversity']
n_contrib_subj = cue_contributor_diversity['n_contributing_subj']

print(f'Contributing children per cue: M = {n_contrib_subj.mean():.1f}, SD = {n_contrib_subj.std(ddof=1):.1f}, range = [{n_contrib_subj.min()}, {n_contrib_subj.max()}]')
print(f'Simpson concentration per cue: M = {simpson_conc.mean():.3f}, SD = {simpson_conc.std(ddof=1):.3f}, range = [{simpson_conc.min():.3f}, {simpson_conc.max():.3f}]')
print(f'Simpson contributor diversity per cue: M = {simpson_div.mean():.3f}, SD = {simpson_div.std(ddof=1):.3f}, range = [{simpson_div.min():.3f}, {simpson_div.max():.3f}]')

print('\nLowest-diversity cues by Simpson contributor diversity:')
print(
    cue_contributor_diversity
    .sort_values(['simpson_contributor_diversity', 'n_associate_contributions'], ascending=[True, False])
    .head(10)
    .round(3)
    .to_string(index=False)
)

cue_contributor_diversity.to_csv(TABLE_DIR / 'cue_contributor_concentration_summary.csv', index=False, encoding='utf-8-sig')

cue_descriptive_rows.extend([
    ['Contributing children per cue: M (SD)', f'{n_contrib_subj.mean():.1f} ({n_contrib_subj.std(ddof=1):.1f})'],
    ['Contributing children per cue range', f'[{n_contrib_subj.min()}, {n_contrib_subj.max()}]'],
    ['Simpson concentration per cue: M (SD)', f'{simpson_conc.mean():.3f} ({simpson_conc.std(ddof=1):.3f})'],
    ['Simpson concentration range', f'[{simpson_conc.min():.3f}, {simpson_conc.max():.3f}]'],
    ['Simpson contributor diversity per cue: M (SD)', f'{simpson_div.mean():.3f} ({simpson_div.std(ddof=1):.3f})'],
    ['Simpson contributor diversity range', f'[{simpson_div.min():.3f}, {simpson_div.max():.3f}]'],
])
pd.DataFrame(cue_descriptive_rows, columns=['Measure', 'Value']).to_csv(
    TABLE_DIR / 'descriptives_cues.csv', index=False, encoding='utf-8-sig')

print('\nSaved: cue_contributor_concentration_summary.csv')
print('Updated: descriptives_cues.csv')

## 3. Set up analysis data

In [ ]:
from __future__ import annotations
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy import stats
from scipy.cluster import hierarchy

OUTPUT_DIR = Path('outputs')
DATA_DIR = OUTPUT_DIR / 'analysis_data'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIG_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

TOKEN_COUNTS_PATH = OUTPUT_COUNTS
RESPONSES_TOKENIZED_PATH = CHILD_INPUT
CUE_SUMMARY_PATH = OUTPUT_SUMMARY

RNG_SEED = 20260806
MAIN_CUE_MIN_NVALID = 20
ASSOCIATION_MIN_REPORT_COUNT = 2
N_RSA_PERMUTATIONS = 1000
NETWORK_ORDER = ['child', 'adult', 'speech', 'book']
SOURCE_LABELS = {'child': 'Child', 'adult': 'Adult', 'speech': 'Speech', 'book': 'Book'}
SOURCE_COLORS = {'child': '#E1B229', 'adult': '#2E6A87', 'speech': '#5F8D78', 'book': '#A9794B'}
CUE_CLASS_ORDER = ['Noun', 'Verb', 'Adj/Adv']
CUE_CLASS_COLORS = {'Noun': '#C9792D', 'Verb': '#6FA1C7', 'Adj/Adv': '#2B8B89'}
plt.rcParams['axes.unicode_minus'] = False

def safe_corr(x: pd.Series, y: pd.Series, method: str) -> float:
    if len(x) < 3:
        return np.nan
    if x.nunique() < 2 or y.nunique() < 2:
        return np.nan
    if method == 'pearson':
        return stats.pearsonr(x, y)[0]
    if method == 'spearman':
        return stats.spearmanr(x, y)[0]
    raise ValueError(method)

token_counts = pd.read_csv(TOKEN_COUNTS_PATH, encoding='utf-8-sig')
token_counts['subj'] = token_counts['subj'].astype(str).str.strip()
token_counts['cue'] = token_counts['cue'].astype(str).str.strip()
token_counts['response'] = token_counts['response'].astype(str).str.strip()
token_counts = token_counts[['subj', 'cue', 'response']].dropna().drop_duplicates().reset_index(drop=True)

cue_meta = pd.read_csv(CUE_SUMMARY_PATH, encoding='utf-8-sig')
cue_meta['cue'] = cue_meta['cue'].astype(str).str.strip()
cue_class_col = 'cue_class_4way_corrected_en' if 'cue_class_4way_corrected_en' in cue_meta.columns else 'cue_class_4way_corrected'
cue_meta = cue_meta[['cue', 'cue_POS', cue_class_col, 'n_subj']].rename(columns={cue_class_col: 'cue_class'})
cue_meta['cue_class'] = cue_meta['cue_class'].replace({'Noun': 'Noun', 'Verb': 'Verb', 'Adj/Adv': 'Adj/Adv', 'Other': 'Other'})

relation_strengths = pd.read_csv(RELATION_PATH, encoding='utf-8-sig')
relation_strengths['cue'] = relation_strengths['cue'].astype(str).str.strip()
relation_strengths['response'] = relation_strengths['response'].astype(str).str.strip()
relation_strengths['source'] = relation_strengths['source'].astype(str).str.strip()
word_measures = pd.read_csv(WORD_MEASURES_PATH, encoding='utf-8-sig')
word_measures['word'] = word_measures['word'].astype(str).str.strip()

adult = relation_strengths.loc[relation_strengths['source'].eq('adult')].copy()
adult = adult.rename(columns={'weight': 'raw_weight', 'aux_weight': 'normalized_weight', 'n_valid_source': 'n_valid_retained'})

env = relation_strengths.loc[relation_strengths['source'].isin(['speech', 'book'])].copy()
env = env.rename(columns={'weight': 'normalized_weight', 'aux_weight': 'raw_weight'})

print(f'child triplets (deduplicated subj-cue-response): {len(token_counts):,}')
print(f'child cues: {token_counts["cue"].nunique():,}')
print('child cue classes:')
print(cue_meta['cue_class'].value_counts().reindex(CUE_CLASS_ORDER).fillna(0).astype(int).to_string())
print(f'adult derived rows: {len(adult):,}')
print(f'environment derived rows: {len(env):,}')

common_cues_4way = sorted(set(token_counts['cue']) & set(adult['cue']) & set(env['cue']))
print(f'common cues across child/adult/speech/book derived inputs: {len(common_cues_4way):,}')


## 4. Child association strengths

In [ ]:
def build_child_association_strength(triplets: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    trip = triplets[['subj', 'cue', 'response']].drop_duplicates().copy()
    cue_summary = (
        trip.groupby('cue', as_index=False)
        .agg(
            n_valid=('subj', 'nunique'),
            cue_total_response_tokens=('response', 'size'),
            n_unique_responses=('response', 'nunique'),
        )
    )
    edges = (
        trip.groupby(['cue', 'response'], as_index=False)
        .agg(
            n_children=('subj', 'nunique'),
            edge_count=('subj', 'size'),
        )
        .merge(cue_summary[['cue', 'n_valid', 'cue_total_response_tokens']], on='cue', how='left')
    )
    edges['child_weight'] = edges['n_children'] / edges['n_valid']
    edges = edges.sort_values(['cue', 'child_weight', 'response'], ascending=[True, False, True]).reset_index(drop=True)
    return edges, cue_summary

child_edges_full, child_cue_summary = build_child_association_strength(token_counts)
child_edges_full.head()


In [ ]:
child_edges_full.to_csv(DATA_DIR / 'child_association_strength_full.csv', index=False)
child_cue_summary.to_csv(TABLE_DIR / 'child_cue_summary_for_association_validation.csv', index=False)

print(f'child cue-response pairs: {len(child_edges_full):,}')
print(f'median n_valid per cue: {child_cue_summary["n_valid"].median():.1f}')
print(f'median cue_total_response_tokens: {child_cue_summary["cue_total_response_tokens"].median():.1f}')


## 5. Word-level lexical availability

Child production frequency is compared with frequency and contextual-diversity measures from child-directed speech and book corpora.

In [ ]:
word_frequency_validation = word_measures.copy()

freq_cols = ['child_freq_per_million', 'speech_freq_per_million', 'book_freq_per_million', 'speech_log_cd', 'book_log_cd']
for col in freq_cols:
    if col not in word_frequency_validation.columns:
        word_frequency_validation[col] = np.nan
    word_frequency_validation[col] = pd.to_numeric(word_frequency_validation[col], errors='coerce')
    if col.endswith('log_cd'):
        word_frequency_validation[f'metric_{col}'] = word_frequency_validation[col]
    else:
        word_frequency_validation[f'metric_{col}'] = np.log1p(word_frequency_validation[col])

word_frequency_validation.to_csv(TABLE_DIR / 'word_level_frequency_validation.csv', index=False)

matrix_cols = [f'metric_{col}' for col in freq_cols]
matrix_labels = {
    'metric_child_freq_per_million': 'CCLAP frequency',
    'metric_speech_freq_per_million': 'Speech frequency',
    'metric_book_freq_per_million': 'Book frequency',
    'metric_speech_log_cd': 'Speech CD',
    'metric_book_log_cd': 'Book CD',
}
matrix_presence_cols = {
    'metric_child_freq_per_million': 'child_freq_per_million',
    'metric_speech_freq_per_million': 'speech_freq_per_million',
    'metric_book_freq_per_million': 'book_freq_per_million',
    'metric_speech_log_cd': 'speech_cd',
    'metric_book_log_cd': 'book_cd',
}

pair_rows = []
word_freq_corr_matrix = pd.DataFrame(
    np.eye(len(matrix_cols)),
    index=[matrix_labels[col] for col in matrix_cols],
    columns=[matrix_labels[col] for col in matrix_cols],
)
word_freq_n_matrix = pd.DataFrame(
    np.nan,
    index=[matrix_labels[col] for col in matrix_cols],
    columns=[matrix_labels[col] for col in matrix_cols],
)
for i, left in enumerate(matrix_cols):
    left_presence = matrix_presence_cols[left]
    word_freq_n_matrix.loc[matrix_labels[left], matrix_labels[left]] = int(word_frequency_validation[left_presence].notna().sum())
    for right in matrix_cols[i + 1:]:
        right_presence = matrix_presence_cols[right]
        sub = word_frequency_validation.loc[
            word_frequency_validation[left_presence].notna()
            & word_frequency_validation[right_presence].notna(),
            ['word', left, right],
        ].dropna()
        rho = safe_corr(sub[left], sub[right], method='spearman')
        n = len(sub)
        word_freq_corr_matrix.loc[matrix_labels[left], matrix_labels[right]] = rho
        word_freq_corr_matrix.loc[matrix_labels[right], matrix_labels[left]] = rho
        word_freq_n_matrix.loc[matrix_labels[left], matrix_labels[right]] = n
        word_freq_n_matrix.loc[matrix_labels[right], matrix_labels[left]] = n
        pair_rows.append({
            'left': matrix_labels[left],
            'right': matrix_labels[right],
            'n_words': n,
            'spearman_rho': rho,
        })
word_freq_pairwise = pd.DataFrame(pair_rows)

word_freq_corr_matrix.to_csv(TABLE_DIR / 'word_level_frequency_spearman_matrix.csv')
word_freq_n_matrix.to_csv(TABLE_DIR / 'word_level_frequency_shared_n_matrix.csv')
word_freq_pairwise.to_csv(TABLE_DIR / 'word_level_frequency_pairwise_spearman.csv', index=False)

print(f'child response corpus tokens before vocabulary/POS screening: {int(word_frequency_validation["child_token_count"].sum()):,}')
print(f'child response word types before vocabulary/POS screening: {word_frequency_validation["child_token_count"].notna().sum():,}')
print('Pairwise Spearman correlations among shared word types:')
print(word_freq_pairwise.round(3).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 6.4))
lower_triangle_mask = np.triu(np.ones_like(word_freq_corr_matrix, dtype=bool), k=1)
sns.heatmap(
    word_freq_corr_matrix,
    mask=lower_triangle_mask,
    vmin=0,
    vmax=1,
    center=0,
    cmap='vlag',
    annot=False,
    square=True,
    linewidths=0.8,
    cbar_kws={'label': 'Spearman rho', 'shrink': 0.45, 'aspect': 12, 'pad': 0.03},
    ax=ax,
)
for i in range(word_freq_corr_matrix.shape[0]):
    for j in range(word_freq_corr_matrix.shape[1]):
        if not lower_triangle_mask[i, j]:
            ax.text(
                j + 0.5,
                i + 0.5,
                f'{word_freq_corr_matrix.iat[i, j]:.2f}',
                ha='center',
                va='center',
                color='black',
                fontsize=11,
            )
            
ax.tick_params(axis='y', rotation=0, labelsize=11)
ax.tick_params(axis='x', rotation=35, labelsize=11)
for label in ax.get_xticklabels():
    label.set_ha('right')
    label.set_x(label.get_position()[0] + 0.08)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=10)
cbar.set_label('Spearman rho', fontsize=11)

fig.tight_layout()
fig.savefig(FIG_DIR / 'word_level_frequency_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Top-associate stability

Top-associate retention is estimated by bootstrapping children within each cue.

In [ ]:
N_BOOT_TOPK = 1000
TOPK_STABILITY_VALUES = list(range(1, 20, 2))
TOPK_FOCAL_K = 9


def topk_p_to_apa(p_value: float) -> str:
    if pd.isna(p_value):
        return '= NA'
    if p_value < 0.001:
        return '< .001'
    return f'= {p_value:.3f}'.replace('0.', '.')


def topk_stars_from_p(p_value: float) -> str:
    if pd.isna(p_value):
        return 'n.s.'
    if p_value < 0.001:
        return '***'
    if p_value < 0.01:
        return '**'
    if p_value < 0.05:
        return '*'
    return 'n.s.'


def compute_topk_bootstrap_retention(
    triplets: pd.DataFrame,
    topks: list[int],
    n_boot: int,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    rows = []
    grouped = triplets[['subj', 'cue', 'response']].drop_duplicates().groupby('cue', sort=True)

    for cue, cue_df in grouped:
        subjects = sorted(cue_df['subj'].unique())
        responses = sorted(cue_df['response'].unique())
        n_valid = len(subjects)
        n_unique = len(responses)
        if n_valid == 0 or n_unique == 0:
            continue

        subj_index = {subj: i for i, subj in enumerate(subjects)}
        resp_index = {resp: i for i, resp in enumerate(responses)}
        matrix = np.zeros((n_valid, n_unique), dtype=np.int16)
        for row in cue_df.itertuples(index=False):
            matrix[subj_index[row.subj], resp_index[row.response]] = 1

        full_counts = matrix.sum(axis=0)
        full_order = sorted(range(n_unique), key=lambda j: (-full_counts[j], responses[j]))
        full_top = {k: set(full_order[:min(k, n_unique)]) for k in topks}
        chance = {k: min(k, n_unique) / n_unique for k in topks}

        sampled_subjects = rng.integers(0, n_valid, size=(n_boot, n_valid))
        bootstrap_counts = np.zeros((n_boot, n_unique), dtype=np.int16)
        for boot_i in range(n_boot):
            bootstrap_counts[boot_i] = matrix[sampled_subjects[boot_i]].sum(axis=0)

        for boot_i in range(n_boot):
            boot_order = sorted(range(n_unique), key=lambda j: (-bootstrap_counts[boot_i, j], responses[j]))
            for k in topks:
                effective_k = min(k, n_unique)
                boot_top = set(boot_order[:effective_k])
                retention = len(full_top[k] & boot_top) / effective_k if effective_k else np.nan
                rows.append({
                    'cue': cue,
                    'bootstrap': boot_i + 1,
                    'top_k': k,
                    'effective_k': effective_k,
                    'n_valid': n_valid,
                    'n_unique_associates': n_unique,
                    'retention': retention,
                    'chance_retention': chance[k],
                })

    retention = pd.DataFrame(rows)
    cue_summary = (
        retention.groupby(['cue', 'top_k'], as_index=False)
        .agg(
            mean_retention=('retention', 'mean'),
            sd_retention=('retention', 'std'),
            chance_retention=('chance_retention', 'mean'),
            n_valid=('n_valid', 'first'),
            n_unique_associates=('n_unique_associates', 'first'),
        )
    )
    summary = (
        cue_summary.groupby('top_k', as_index=False)
        .agg(
            mean_retention=('mean_retention', 'mean'),
            sd_across_cues=('mean_retention', 'std'),
            median_retention=('mean_retention', 'median'),
            q025=('mean_retention', lambda s: s.quantile(0.025)),
            q975=('mean_retention', lambda s: s.quantile(0.975)),
            mean_chance=('chance_retention', 'mean'),
            n_cues=('cue', 'nunique'),
        )
    )
    summary['se_across_cues'] = summary['sd_across_cues'] / np.sqrt(summary['n_cues'])
    summary['ci_low'] = summary['mean_retention'] - 1.96 * summary['se_across_cues']
    summary['ci_high'] = summary['mean_retention'] + 1.96 * summary['se_across_cues']
    return cue_summary, summary


topk_cue_summary, topk_retention_summary = compute_topk_bootstrap_retention(
    token_counts,
    topks=TOPK_STABILITY_VALUES,
    n_boot=N_BOOT_TOPK,
    seed=RNG_SEED,
)

topk_cue_summary.to_csv(TABLE_DIR / 'child_topk_bootstrap_retention_by_cue.csv', index=False)
topk_retention_summary.to_csv(TABLE_DIR / 'child_topk_bootstrap_retention_summary.csv', index=False)

focal = topk_cue_summary.loc[topk_cue_summary['top_k'] == TOPK_FOCAL_K].copy()
focal_rho, focal_p = stats.spearmanr(focal['n_valid'], focal['mean_retention'])

fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.15))
for ax, panel_label in zip(axes, ['A', 'B']):
    ax.text(
        -0.12, 1.06, panel_label,
        transform=ax.transAxes,
        ha='left', va='bottom',
        fontsize=15, fontweight='bold',
    )

ax = axes[0]
x = topk_retention_summary['top_k'].to_numpy(float)
y = topk_retention_summary['mean_retention'].to_numpy(float)
sd = topk_retention_summary['sd_across_cues'].to_numpy(float)
lo = np.clip(y - sd, 0, 1)
hi = np.clip(y + sd, 0, 1)
chance = topk_retention_summary['mean_chance'].to_numpy(float)
yerr = np.vstack([y - lo, hi - y])
dodge = 0.16
ax.errorbar(
    x - dodge, y, yerr=yerr,
    fmt='o', markersize=5.8,
    color=SOURCE_COLORS['child'], ecolor=SOURCE_COLORS['child'],
    elinewidth=1.7, capsize=3.2, capthick=1.4,
    markeredgecolor='white', markeredgewidth=0.7,
    label='Observed retention', zorder=3,
)
ax.scatter(
    x + dodge, chance,
    s=32, facecolors='white', edgecolors='0.42', linewidths=1.35,
    label='Chance baseline', zorder=2,
)
ax.set_xticks(x)
ax.set_xlim(0, 20)
ax.set_ylim(0, 1.02)
ax.set_xlabel('Top-k associates', fontsize=14)
ax.set_ylabel('Bootstrap retention proportion', fontsize=14)
ax.tick_params(axis='both', labelsize=12)
ax.legend(
    frameon=False, loc='upper right', bbox_to_anchor=(0.98, 0.98),
    fontsize=10.8, handlelength=1.2, labelspacing=0.55, borderaxespad=0.0,
)

ax = axes[1]
ax.scatter(
    focal['n_valid'], focal['mean_retention'],
    s=38, color=SOURCE_COLORS['child'], alpha=0.7, edgecolor='none', zorder=2,
)
x_values = focal['n_valid'].to_numpy(float)
y_values = focal['mean_retention'].to_numpy(float)
slope, intercept, linear_r, linear_p, linear_se = stats.linregress(x_values, y_values)
x_grid = np.linspace(x_values.min(), x_values.max(), 160)
y_hat = intercept + slope * x_grid
n = len(x_values)
x_bar = x_values.mean()
ss_x = np.sum((x_values - x_bar) ** 2)
residuals = y_values - (intercept + slope * x_values)
s_err = np.sqrt(np.sum(residuals ** 2) / (n - 2))
t_crit = stats.t.ppf(0.975, n - 2)
ci = t_crit * s_err * np.sqrt(1 / n + (x_grid - x_bar) ** 2 / ss_x)
ax.plot(x_grid, y_hat, color=SOURCE_COLORS['child'], linewidth=2.2, zorder=3)
ax.fill_between(x_grid, y_hat - ci, y_hat + ci, color=SOURCE_COLORS['child'], alpha=0.18, linewidth=0, zorder=1)
ax.set_xlabel('N valid responses', fontsize=14)
ax.set_ylabel(f'Top-{TOPK_FOCAL_K} retention proportion', fontsize=14)
ax.set_xlim(x_values.min() - 1, x_values.max() + 1)
ax.set_ylim(0, 1.02)
ax.tick_params(axis='both', labelsize=12)
ax.text(
    0.95, 0.05, f'rho = {focal_rho:.2f}{topk_stars_from_p(focal_p)}',
    transform=ax.transAxes,
    ha='right', va='bottom', fontsize=13,
    color=SOURCE_COLORS['child'], fontweight='bold',
)

for ax in axes:
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

fig.tight_layout(w_pad=0.85)
fig.savefig(FIG_DIR / 'child_topk_bootstrap_retention.png', dpi=300, bbox_inches='tight')
plt.show()

print('Top-associate bootstrap retention summary:')
display(topk_retention_summary.round(4))
print(f'Top-{TOPK_FOCAL_K} retention vs N valid responses: rho = {focal_rho:.3f}, p {topk_p_to_apa(focal_p)}')




## 7. Child-adult association entropy

In [ ]:
# ---- cue-level association entropy: child vs adult, associate-count matched within cue ----
def shannon_entropy_from_weights(weights: pd.Series) -> tuple[float, float]:
    weights = pd.to_numeric(weights, errors='coerce').dropna()
    weights = weights.loc[weights > 0]
    if len(weights) == 0:
        return np.nan, np.nan
    probs = weights / weights.sum()
    entropy = float(-(probs * np.log2(probs)).sum())
    normalized_entropy = entropy / np.log2(len(probs)) if len(probs) > 1 else np.nan
    return entropy, normalized_entropy


def top_k_by_strength(group: pd.DataFrame, weight_col: str, k: int) -> pd.DataFrame:
    return (
        group.sort_values([weight_col, 'response'], ascending=[False, True])
        .head(k)
        .copy()
    )


def cue_matched_entropy_table(child_pairs: pd.DataFrame,
                              adult_pairs: pd.DataFrame,
                              cues: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    detail_rows = []
    for cue in cues:
        child_group = child_pairs.loc[child_pairs['cue'].eq(cue), ['cue', 'response', 'child_weight']].copy()
        adult_group = adult_pairs.loc[adult_pairs['cue'].eq(cue), ['cue', 'response', 'adult_weight']].copy()
        child_group = child_group.loc[pd.to_numeric(child_group['child_weight'], errors='coerce').gt(0)].copy()
        adult_group = adult_group.loc[pd.to_numeric(adult_group['adult_weight'], errors='coerce').gt(0)].copy()
        child_n = child_group['response'].nunique()
        adult_n = adult_group['response'].nunique()
        matched_k = int(min(child_n, adult_n))
        if matched_k < 2:
            continue

        child_top = top_k_by_strength(child_group, 'child_weight', matched_k)
        adult_top = top_k_by_strength(adult_group, 'adult_weight', matched_k)
        for source, top_df, weight_col, original_n in [
            ('child', child_top, 'child_weight', child_n),
            ('adult', adult_top, 'adult_weight', adult_n),
        ]:
            entropy, normalized_entropy = shannon_entropy_from_weights(top_df[weight_col])
            rows.append({
                'cue': cue,
                'source': source,
                'n_associates_original': original_n,
                'matched_k': matched_k,
                'shannon_entropy': entropy,
                'normalized_entropy': normalized_entropy,
            })
            detail = top_df.rename(columns={weight_col: 'weight'})[['cue', 'response', 'weight']].copy()
            detail['source'] = source
            detail['matched_k'] = matched_k
            detail['n_associates_original'] = original_n
            detail['rank_within_source'] = np.arange(1, len(detail) + 1)
            detail_rows.append(detail)
    detail_df = pd.concat(detail_rows, ignore_index=True) if detail_rows else pd.DataFrame()
    return pd.DataFrame(rows), detail_df


entropy_cues = sorted(
    set(child_cue_summary.loc[child_cue_summary['n_valid'] >= MAIN_CUE_MIN_NVALID, 'cue'])
    & set(adult['cue'])
)
adult_entropy_pairs = (
    adult[['cue', 'response', 'raw_weight']]
    .dropna(subset=['cue', 'response', 'raw_weight'])
    .copy()
)
adult_entropy_pairs['adult_weight'] = pd.to_numeric(adult_entropy_pairs['raw_weight'], errors='coerce')
adult_entropy_pairs = adult_entropy_pairs.loc[adult_entropy_pairs['adult_weight'] > 0].copy()

association_entropy, association_entropy_matched_profiles = cue_matched_entropy_table(
    child_edges_full,
    adult_entropy_pairs,
    entropy_cues,
)
association_entropy.to_csv(TABLE_DIR / 'child_adult_association_entropy.csv', index=False)
association_entropy_matched_profiles.to_csv(
    TABLE_DIR / 'child_adult_association_entropy_matched_profiles.csv',
    index=False,
)

entropy_wide = (
    association_entropy
    .pivot(index='cue', columns='source', values='normalized_entropy')
    .dropna()
    .reset_index()
)
entropy_ttest = stats.ttest_rel(entropy_wide['child'], entropy_wide['adult'])
entropy_summary = (
    association_entropy
    .groupby('source', as_index=False)
    .agg(
        n_cues=('cue', 'nunique'),
        mean_normalized_entropy=('normalized_entropy', 'mean'),
        sd_normalized_entropy=('normalized_entropy', 'std'),
        mean_shannon_entropy=('shannon_entropy', 'mean'),
        sd_shannon_entropy=('shannon_entropy', 'std'),
        mean_n_associates_original=('n_associates_original', 'mean'),
        sd_n_associates_original=('n_associates_original', 'std'),
        mean_matched_k=('matched_k', 'mean'),
        sd_matched_k=('matched_k', 'std'),
    )
)
entropy_summary.to_csv(TABLE_DIR / 'child_adult_association_entropy_summary.csv', index=False)

print('Cue-level normalized association entropy after cue-wise associate-count matching:')
display(entropy_summary.round(3))
print(
    f"Paired t-test, child vs adult matched normalized entropy: "
    f"t({len(entropy_wide) - 1}) = {entropy_ttest.statistic:.2f}, p = {entropy_ttest.pvalue:.3g}"
)

entropy_plot = association_entropy.dropna(subset=['normalized_entropy']).copy()
source_order = ['child', 'adult']
summary_plot = (
    entropy_plot
    .groupby('source', as_index=False)
    .agg(
        mean=('normalized_entropy', 'mean'),
        sem=('normalized_entropy', 'sem'),
        n=('normalized_entropy', 'size'),
    )
)
summary_plot['ci95'] = summary_plot['sem'] * 1.96

fig, ax = plt.subplots(figsize=(3.8, 3.6))
x_positions = {'child': 0, 'adult': 1}
rng = np.random.default_rng(RNG_SEED)
for source in source_order:
    data = entropy_plot.loc[entropy_plot['source'].eq(source), 'normalized_entropy'].to_numpy()
    x = x_positions[source]
    jitter = rng.normal(0, 0.045, size=len(data))
    ax.scatter(
        np.full(len(data), x) + jitter,
        data,
        s=9,
        alpha=0.28,
        color=SOURCE_COLORS[source],
        edgecolor='none',
        zorder=1,
    )
    row = summary_plot.loc[summary_plot['source'].eq(source)].iloc[0]
    ax.bar(
        x,
        row['mean'],
        width=0.52,
        color=SOURCE_COLORS[source],
        alpha=0.82,
        edgecolor='#2F2F2F',
        linewidth=0.8,
        zorder=2,
    )
    ax.errorbar(
        x,
        row['mean'],
        yerr=row['ci95'],
        color='#2F2F2F',
        capsize=3,
        linewidth=1.0,
        zorder=3,
    )
    ax.text(
        x,
        row['mean'] + row['ci95'] + 0.006,
        f"H = {row['mean']:.3f}",
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight='bold',
        color='#2F2F2F',
    )
ax.set_xticks([0, 1])
ax.set_xticklabels(['Child', 'Adult'], fontsize=9.5)
ax.set_ylabel('Matched normalized association entropy', fontsize=10)
ax.set_xlabel('')
ax.set_ylim(
    max(0, entropy_plot['normalized_entropy'].min() - 0.04),
    min(1.02, entropy_plot['normalized_entropy'].max() + 0.04),
)
ax.tick_params(axis='y', labelsize=9)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(FIG_DIR / 'child_adult_association_entropy.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: child_adult_association_entropy.png')
print('Saved: child_adult_association_entropy.csv')
print('Saved: child_adult_association_entropy_summary.csv')
print('Saved: child_adult_association_entropy_matched_profiles.csv')


In [ ]:
# ---- example cue profile for association entropy ----
EXAMPLE_ENTROPY_CUE = '柠檬'
EXAMPLE_CONTENT_POS = {'NN', 'NR', 'NT', 'VV', 'VC', 'VE', 'JJ', 'VA', 'AD'}
EXAMPLE_DISPLAY_EXCLUDE_WORDS = {'很', '就', '不', '一', '可以', '在', '还', '能', '想到'}
EXAMPLE_TOP_N = 15


def response_pos_lookup_from_tokenized(raw_resp: pd.DataFrame) -> pd.DataFrame:
    rows = []
    token_col = 'segmented_tokens_with_pos' if 'segmented_tokens_with_pos' in raw_resp.columns else 'tokens_with_pos'
    for tokens_with_pos in raw_resp[token_col].dropna():
        rows.extend(parse_token_string(tokens_with_pos))
    response_pos = pd.DataFrame(rows, columns=['response', 'pos']).drop_duplicates()
    response_pos = (
        response_pos
        .groupby('response', as_index=False)
        .agg(pos_tags=('pos', lambda x: sorted(set(x))))
    )
    response_pos['keep_content_for_display'] = response_pos['pos_tags'].apply(
        lambda tags: any(tag in EXAMPLE_CONTENT_POS for tag in tags)
    )
    return response_pos


def top_example_profile(pairs: pd.DataFrame,
                        cue: str,
                        source: str,
                        response_pos: pd.DataFrame | None = None,
                        top_n: int = EXAMPLE_TOP_N) -> tuple[pd.DataFrame, float]:
    if source == 'child':
        profile = (
            pairs.loc[pairs['cue'].eq(cue), ['response', 'child_weight']]
            .rename(columns={'child_weight': 'weight'})
            .copy()
        )
        if response_pos is not None:
            profile = profile.merge(response_pos[['response', 'keep_content_for_display']], on='response', how='left')
            profile = profile.loc[profile['keep_content_for_display'].fillna(False)].copy()
    elif source == 'adult':
        profile = (
            pairs.loc[pairs['cue'].eq(cue), ['response', 'adult_weight']]
            .rename(columns={'adult_weight': 'weight'})
            .copy()
        )
    else:
        raise ValueError(f'Unknown source: {source}')

    profile = profile.loc[~profile['response'].isin(EXAMPLE_DISPLAY_EXCLUDE_WORDS)].copy()
    profile = profile.groupby('response', as_index=False)['weight'].sum()
    profile = profile.loc[pd.to_numeric(profile['weight'], errors='coerce').gt(0)].copy()
    profile['prop'] = profile['weight'] / profile['weight'].sum()
    entropy = shannon_entropy_from_weights(profile['weight'])[1]
    return profile.sort_values(['prop', 'response'], ascending=[False, True]).head(top_n), entropy


response_pos_for_example = response_pos_lookup_from_tokenized(pd.read_csv(CHILD_INPUT, encoding='utf-8-sig'))
child_example_profile, child_example_entropy = top_example_profile(
    child_edges_full,
    EXAMPLE_ENTROPY_CUE,
    'child',
    response_pos=response_pos_for_example,
)
adult_example_profile, adult_example_entropy = top_example_profile(
    adult_entropy_pairs,
    EXAMPLE_ENTROPY_CUE,
    'adult',
)

example_ymax = max(
    child_example_profile['prop'].max(),
    adult_example_profile['prop'].max(),
) * 1.18

fig, axes = plt.subplots(2, 1, figsize=(7.1, 4.8), sharey=True)
for ax, profile, title, color, entropy_value in [
    (axes[0], child_example_profile, 'Child associations', SOURCE_COLORS['child'], child_example_entropy),
    (axes[1], adult_example_profile, 'Adult associations', SOURCE_COLORS['adult'], adult_example_entropy),
]:
    x = np.arange(len(profile) + 1)
    labels = list(profile['response']) + ['...']
    values = list(profile['prop']) + [0]
    ax.bar(x, values, color=color, alpha=0.83, width=0.72)
    ax.set_title(title, fontsize=17, fontweight='bold', loc='center', pad=6)
    ax.text(
        0.99,
        0.88,
        f'H = {entropy_value:.3f}',
        transform=ax.transAxes,
        ha='right',
        va='top',
        fontsize=13,
        fontweight='bold',
        color='#2F2F2F',
    )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha='right', rotation_mode='anchor', fontsize=11)
    ax.tick_params(axis='y', labelsize=11, colors='black')
    ax.tick_params(axis='x', pad=1)
    ax.set_ylim(0, example_ymax)
    ax.grid(False)
    sns.despine(ax=ax)

axes[0].set_ylabel('Response proportion', fontsize=14)
axes[1].set_ylabel('Response proportion', fontsize=14)
fig.tight_layout(h_pad=1.45)
fig.savefig(FIG_DIR / 'example_lemon_association_profile_top15_entropy.png', dpi=300, bbox_inches='tight')
plt.show()

print('Child example profile:')
display(child_example_profile[['response', 'prop']].round(3))
print('Adult example profile:')
display(adult_example_profile[['response', 'prop']].round(3))
print('Saved: example_lemon_association_profile_top15_entropy.png')


## 8. Child-adult RSA

The main RSA uses the 467 shared cues with at least 20 valid retained child responses.

In [ ]:
def make_cue_response_matrix(pairs: pd.DataFrame,
                             value_col: str,
                             cues: list[str],
                             responses: list[str]) -> pd.DataFrame:
    mat = (
        pairs.groupby(['cue', 'response'], as_index=False)[value_col].sum()
        .pivot(index='cue', columns='response', values=value_col)
        .reindex(index=cues, columns=responses)
        .fillna(0.0)
    )
    return mat

def row_cosine_similarity(mat: pd.DataFrame) -> pd.DataFrame:
    x = mat.to_numpy(dtype=float)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = np.nan
    x_norm = x / norms
    sim = x_norm @ x_norm.T
    return pd.DataFrame(sim, index=mat.index, columns=mat.index)

def row_pearson_similarity(mat: pd.DataFrame) -> pd.DataFrame:
    sim = np.corrcoef(mat.to_numpy(dtype=float))
    return pd.DataFrame(sim, index=mat.index, columns=mat.index)

def lower_triangle_values(mat: pd.DataFrame | np.ndarray) -> np.ndarray:
    arr = mat.to_numpy() if isinstance(mat, pd.DataFrame) else np.asarray(mat)
    idx = np.tril_indices_from(arr, k=-1)
    return arr[idx]

def spearman_vector(x: np.ndarray, y: np.ndarray) -> float:
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3:
        return np.nan
    x_ok = x[ok]
    y_ok = y[ok]
    if np.nanstd(x_ok) == 0 or np.nanstd(y_ok) == 0:
        return np.nan
    return stats.spearmanr(x_ok, y_ok)[0]

def rsa_spearman(left: pd.DataFrame, right: pd.DataFrame) -> tuple[float, float, int]:
    left_vals = lower_triangle_values(left)
    right_vals = lower_triangle_values(right)
    ok = np.isfinite(left_vals) & np.isfinite(right_vals)
    if ok.sum() < 3:
        return np.nan, np.nan, int(ok.sum())
    rho, p = stats.spearmanr(left_vals[ok], right_vals[ok])
    return rho, p, int(ok.sum())

def rsa_permutation_test(left: pd.DataFrame,
                         right: pd.DataFrame,
                         observed_rho: float,
                         n_perm: int = 1000,
                         seed: int = 20260806) -> tuple[pd.DataFrame, float]:
    rng = np.random.default_rng(seed)
    left_vals = lower_triangle_values(left)
    right_arr = right.to_numpy()
    rows = []
    for perm_id in range(1, n_perm + 1):
        perm = rng.permutation(right_arr.shape[0])
        right_perm = right_arr[np.ix_(perm, perm)]
        right_vals = lower_triangle_values(right_perm)
        rho = spearman_vector(left_vals, right_vals)
        rows.append({'perm_id': perm_id, 'rsa_spearman_rho': rho})
    null_df = pd.DataFrame(rows)
    p_perm_greater = (1 + (null_df['rsa_spearman_rho'] >= observed_rho).sum()) / (len(null_df) + 1)
    return null_df, p_perm_greater

def cue_class_sorted_order(cues: list[str], child_distance: pd.DataFrame) -> list[str]:
    meta = cue_meta.loc[cue_meta['cue'].isin(cues), ['cue', 'cue_class']].copy()
    meta['cue_class'] = pd.Categorical(meta['cue_class'], categories=CUE_CLASS_ORDER, ordered=True)
    ordered = []
    for cue_class in CUE_CLASS_ORDER:
        class_cues = meta.loc[meta['cue_class'] == cue_class, 'cue'].tolist()
        if len(class_cues) <= 2:
            ordered.extend(sorted(class_cues))
            continue
        sub = child_distance.loc[class_cues, class_cues].copy()
        condensed = sub.to_numpy()[np.tril_indices(len(sub), k=-1)]
        if np.isfinite(condensed).sum() < len(condensed):
            ordered.extend(sorted(class_cues))
            continue
        linkage = hierarchy.linkage(condensed, method='average')
        leaves = hierarchy.leaves_list(linkage)
        ordered.extend([class_cues[i] for i in leaves])
    other_cues = sorted(set(cues) - set(ordered))
    return ordered + other_cues

main_analysis_cues = set(child_cue_summary.loc[child_cue_summary['n_valid'] >= MAIN_CUE_MIN_NVALID, 'cue'])
main_analysis_cues &= set(adult['cue'])

child_sub = child_edges_full.loc[child_edges_full['cue'].isin(main_analysis_cues)].copy()
adult_pairs = (
    adult[['cue', 'response', 'raw_weight']]
    .dropna(subset=['cue', 'response', 'raw_weight'])
    .copy()
)
adult_pairs['adult_weight'] = pd.to_numeric(adult_pairs['raw_weight'], errors='coerce')
adult_pairs = adult_pairs.loc[
    adult_pairs['cue'].isin(main_analysis_cues)
    & adult_pairs['adult_weight'].gt(0)
].copy()

rsa_cues = sorted(main_analysis_cues)
rsa_response_vocab = sorted(set(child_sub['response']) | set(adult_pairs['response']))

child_weighted_mat = make_cue_response_matrix(child_sub, 'child_weight', rsa_cues, rsa_response_vocab)
adult_weighted_mat = make_cue_response_matrix(adult_pairs, 'adult_weight', rsa_cues, rsa_response_vocab)
child_binary_mat = (child_weighted_mat > 0).astype(float)
adult_binary_mat = (adult_weighted_mat > 0).astype(float)

child_weighted_similarity = row_cosine_similarity(child_weighted_mat)
adult_weighted_similarity = row_cosine_similarity(adult_weighted_mat)
child_binary_similarity = row_pearson_similarity(child_binary_mat)
adult_binary_similarity = row_pearson_similarity(adult_binary_mat)

rsa_rows = []
rsa_null_dfs = []
for method, child_sim, adult_sim in [
    ('weighted_cosine', child_weighted_similarity, adult_weighted_similarity),
    ('binary_pearson_cox_style', child_binary_similarity, adult_binary_similarity),
]:
    rho, p, n_pairs = rsa_spearman(child_sim, adult_sim)
    null_df, p_perm = rsa_permutation_test(
        child_sim, adult_sim, observed_rho=rho,
        n_perm=N_RSA_PERMUTATIONS, seed=RNG_SEED,
    )
    null_df['method'] = method
    rsa_null_dfs.append(null_df)
    rsa_rows.append({
        'comparison': 'child_vs_adult',
        'method': method,
        'n_cues': len(rsa_cues),
        'n_response_vocab': len(rsa_response_vocab),
        'n_cue_pairs': n_pairs,
        'rsa_spearman_rho': rho,
        'rsa_spearman_p': p,
        'rsa_permutation_p_greater': p_perm,
        'null_mean': null_df['rsa_spearman_rho'].mean(),
        'null_sd': null_df['rsa_spearman_rho'].std(ddof=1),
        'null_min': null_df['rsa_spearman_rho'].min(),
        'null_max': null_df['rsa_spearman_rho'].max(),
        'n_permutations': N_RSA_PERMUTATIONS,
        'min_cleaned_nvalid': MAIN_CUE_MIN_NVALID,
    })

child_adult_rsa_summary = pd.DataFrame(rsa_rows)
child_adult_rsa_permutation_null = pd.concat(rsa_null_dfs, ignore_index=True)
child_adult_rsa_summary.to_csv(TABLE_DIR / 'child_adult_rsa_summary.csv', index=False)
child_adult_rsa_permutation_null.to_csv(TABLE_DIR / 'child_adult_rsa_permutation_null.csv', index=False)
child_weighted_similarity.to_csv(TABLE_DIR / 'child_rsa_weighted_cosine_similarity_matrix.csv')
adult_weighted_similarity.to_csv(TABLE_DIR / 'adult_rsa_weighted_cosine_similarity_matrix.csv')
child_binary_similarity.to_csv(TABLE_DIR / 'child_rsa_binary_pearson_similarity_matrix.csv')
adult_binary_similarity.to_csv(TABLE_DIR / 'adult_rsa_binary_pearson_similarity_matrix.csv')

print(f'RSA cues: {len(rsa_cues):,}')
print(f'Full response vocabulary: {len(rsa_response_vocab):,}')
child_adult_rsa_summary


In [ ]:
def top_k_associates_by_cue(df: pd.DataFrame,
                              weight_col: str,
                              k_lookup: dict[str, int]) -> pd.DataFrame:
    parts = []
    for cue, sub in df.groupby('cue', sort=True):
        k = int(k_lookup.get(cue, 0))
        if k <= 0:
            continue
        parts.append(
            sub.sort_values([weight_col, 'response'], ascending=[False, True]).head(k)
        )
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[0:0].copy()

adult_associate_counts = adult_pairs.groupby('cue')['response'].nunique().reindex(rsa_cues, fill_value=0)
child_associate_counts = child_sub.groupby('cue')['response'].nunique().reindex(rsa_cues, fill_value=0)
matched_k_by_cue = np.minimum(child_associate_counts, adult_associate_counts).astype(int)
matched_k_lookup = matched_k_by_cue.to_dict()

child_adult_rsa_matched_counts = pd.DataFrame({
    'cue': rsa_cues,
    'child_n_associates': child_associate_counts.values,
    'adult_n_associates': adult_associate_counts.values,
    'matched_k': matched_k_by_cue.values,
})
child_adult_rsa_matched_counts.to_csv(
    TABLE_DIR / 'child_adult_rsa_cuewise_associate_count_matched_counts.csv',
    index=False,
)

child_matched_assoc = top_k_associates_by_cue(child_sub, 'child_weight', matched_k_lookup)
adult_matched_assoc = top_k_associates_by_cue(adult_pairs, 'adult_weight', matched_k_lookup)
matched_response_vocab = sorted(set(child_matched_assoc['response']) | set(adult_matched_assoc['response']))

child_matched_weighted_mat = make_cue_response_matrix(
    child_matched_assoc, 'child_weight', rsa_cues, matched_response_vocab)
adult_matched_weighted_mat = make_cue_response_matrix(
    adult_matched_assoc, 'adult_weight', rsa_cues, matched_response_vocab)
child_matched_binary_mat = (child_matched_weighted_mat > 0).astype(float)
adult_matched_binary_mat = (adult_matched_weighted_mat > 0).astype(float)

matched_rsa_rows = []
matched_rsa_null_dfs = []
for method, child_sim, adult_sim in [
    ('weighted_cosine', row_cosine_similarity(child_matched_weighted_mat), row_cosine_similarity(adult_matched_weighted_mat)),
    ('binary_pearson_cox_style', row_pearson_similarity(child_matched_binary_mat), row_pearson_similarity(adult_matched_binary_mat)),
]:
    rho, p, n_pairs = rsa_spearman(child_sim, adult_sim)
    null_df, p_perm = rsa_permutation_test(
        child_sim, adult_sim, observed_rho=rho,
        n_perm=N_RSA_PERMUTATIONS, seed=RNG_SEED,
    )
    null_df['method'] = method
    matched_rsa_null_dfs.append(null_df)
    matched_rsa_rows.append({
        'comparison': 'child_vs_adult',
        'analysis': 'cuewise_associate_count_matched_top_strength',
        'method': method,
        'n_cues': len(rsa_cues),
        'n_response_vocab': len(matched_response_vocab),
        'n_cue_pairs': n_pairs,
        'child_associates_mean': child_associate_counts.mean(),
        'child_associates_sd': child_associate_counts.std(ddof=1),
        'child_associates_min': child_associate_counts.min(),
        'child_associates_max': child_associate_counts.max(),
        'adult_associates_mean': adult_associate_counts.mean(),
        'adult_associates_sd': adult_associate_counts.std(ddof=1),
        'adult_associates_min': adult_associate_counts.min(),
        'adult_associates_max': adult_associate_counts.max(),
        'matched_k_mean': matched_k_by_cue.mean(),
        'matched_k_sd': matched_k_by_cue.std(ddof=1),
        'matched_k_min': matched_k_by_cue.min(),
        'matched_k_max': matched_k_by_cue.max(),
        'rsa_spearman_rho': rho,
        'rsa_spearman_p': p,
        'rsa_permutation_p_greater': p_perm,
        'null_mean': null_df['rsa_spearman_rho'].mean(),
        'null_sd': null_df['rsa_spearman_rho'].std(ddof=1),
        'null_min': null_df['rsa_spearman_rho'].min(),
        'null_max': null_df['rsa_spearman_rho'].max(),
        'n_permutations': N_RSA_PERMUTATIONS,
        'min_cleaned_nvalid': MAIN_CUE_MIN_NVALID,
    })

child_adult_rsa_cuewise_associate_count_matched = pd.DataFrame(matched_rsa_rows)
child_adult_rsa_cuewise_associate_count_matched_null = pd.concat(matched_rsa_null_dfs, ignore_index=True)
child_adult_rsa_cuewise_associate_count_matched.to_csv(
    TABLE_DIR / 'child_adult_rsa_cuewise_associate_count_matched_summary.csv',
    index=False,
)
child_adult_rsa_cuewise_associate_count_matched_null.to_csv(
    TABLE_DIR / 'child_adult_rsa_cuewise_associate_count_matched_permutation_null.csv',
    index=False,
)
child_matched_weighted_mat.to_csv(TABLE_DIR / 'child_rsa_cuewise_associate_count_matched_weight_matrix.csv')
adult_matched_weighted_mat.to_csv(TABLE_DIR / 'adult_rsa_cuewise_associate_count_matched_weight_matrix.csv')

print('Adult associates per cue in the main RSA cue set:')
print(f'M = {adult_associate_counts.mean():.1f}, SD = {adult_associate_counts.std(ddof=1):.1f}, range = [{adult_associate_counts.min()}, {adult_associate_counts.max()}]')
print('Child associates per cue in the main RSA cue set:')
print(f'M = {child_associate_counts.mean():.1f}, SD = {child_associate_counts.std(ddof=1):.1f}, range = [{child_associate_counts.min()}, {child_associate_counts.max()}]')
print('Matched k per cue:')
print(f'M = {matched_k_by_cue.mean():.1f}, SD = {matched_k_by_cue.std(ddof=1):.1f}, range = [{matched_k_by_cue.min()}, {matched_k_by_cue.max()}]')
print(f'Cues with adult > child associate counts: {(adult_associate_counts > child_associate_counts).sum()}')
print(f'Cues with child > adult associate counts: {(child_associate_counts > adult_associate_counts).sum()}')
child_adult_rsa_cuewise_associate_count_matched


In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

def p_to_stars(p: float) -> str:
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''

def rank_normalize_distance(dist: pd.DataFrame, order: pd.Index) -> np.ndarray:
    arr = dist.loc[order, order].to_numpy(dtype=float)
    out = np.zeros_like(arr, dtype=float)
    lower = np.tril_indices_from(arr, k=-1)
    vals = arr[lower]
    ok = np.isfinite(vals)
    ranks = pd.Series(vals[ok]).rank(method='average').to_numpy()
    scaled = (ranks - 1) / max(len(ranks) - 1, 1)
    out_vals = np.zeros(vals.shape, dtype=float)
    out_vals[ok] = scaled
    out[lower] = out_vals
    out[(lower[1], lower[0])] = out_vals
    np.fill_diagonal(out, 0.0)
    return out

rsa_figure_methods = [
    ('binary_pearson_cox_style', 'Unweighted', child_binary_similarity, adult_binary_similarity),
    ('weighted_cosine', 'Weighted', child_weighted_similarity, adult_weighted_similarity),
]

fig = plt.figure(figsize=(10.4, 7.5))
gs = fig.add_gridspec(
    3, 4,
    width_ratios=[1, 0.16, 1, 1.08],
    height_ratios=[1, 1, 0.045],
    wspace=0.22,
    hspace=0.26,
)
last_im = None

shared_order_distance = (1 - child_binary_similarity).clip(lower=0)
np.fill_diagonal(shared_order_distance.values, 0)
shared_condensed_distance = squareform(shared_order_distance.to_numpy(), checks=False)
rsa_plot_order = shared_order_distance.index[
    leaves_list(linkage(shared_condensed_distance, method='average'))
]

for row_i, (method, method_label, child_similarity, adult_similarity) in enumerate(rsa_figure_methods):
    child_distance = (1 - child_similarity).clip(lower=0)
    adult_distance = (1 - adult_similarity).clip(lower=0)
    np.fill_diagonal(child_distance.values, 0)
    np.fill_diagonal(adult_distance.values, 0)

    ax_child = fig.add_subplot(gs[row_i, 0])
    ax_mid = fig.add_subplot(gs[row_i, 1])
    ax_adult = fig.add_subplot(gs[row_i, 2])
    ax_dist = fig.add_subplot(gs[row_i, 3])

    last_im = ax_child.imshow(rank_normalize_distance(child_distance, rsa_plot_order), cmap='mako_r', vmin=0, vmax=1, interpolation='nearest')
    ax_adult.imshow(rank_normalize_distance(adult_distance, rsa_plot_order), cmap='mako_r', vmin=0, vmax=1, interpolation='nearest')

    for ax, title in [(ax_child, f'Child {method_label} RDM'), (ax_adult, f'Adult {method_label} RDM')]:
        ax.set_title(title, fontsize=12, pad=6)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    main_row = child_adult_rsa_summary.loc[child_adult_rsa_summary['method'] == method].iloc[0]
    ax_mid.set_axis_off()
    ax_mid.annotate(
        '', xy=(0.92, 0.50), xytext=(0.08, 0.50),
        xycoords='axes fraction', textcoords='axes fraction',
        arrowprops=dict(arrowstyle='<->', lw=1.5, color='#333333'),
    )
    ax_mid.text(0.50, 0.62, 'RSA', ha='center', va='bottom', fontsize=11, fontweight='bold', transform=ax_mid.transAxes)
    ax_mid.text(
        0.50, 0.38,
        f"rho = {main_row['rsa_spearman_rho']:.3f}{p_to_stars(main_row['rsa_permutation_p_greater'])}",
        ha='center', va='top', fontsize=10, transform=ax_mid.transAxes,
    )

    null_plot = child_adult_rsa_permutation_null.loc[child_adult_rsa_permutation_null['method'] == method]
    sns.histplot(null_plot['rsa_spearman_rho'], bins=30, color='#B8C0C8', edgecolor='white', ax=ax_dist)
    ax_dist.axvline(main_row['rsa_spearman_rho'], color='#C33A32', linestyle='--', linewidth=2)
    ax_dist.set_title('Permutation null' if row_i == 0 else '', fontsize=12, pad=6)
    ax_dist.set_xlabel('RSA rho' if row_i == 1 else '', fontsize=11)
    ax_dist.set_ylabel('')
    sns.despine(ax=ax_dist)

    ax_dist.tick_params(axis='both', labelsize=10)

    ax_child.text(-0.10, 1.08, 'A' if row_i == 0 else 'B', transform=ax_child.transAxes, ha='left', va='bottom', fontsize=15, fontweight='bold')

cax = fig.add_subplot(gs[2, 0:3])
fig.colorbar(last_im, cax=cax, orientation='horizontal', label='Rank-normalized distance')
cax.tick_params(axis='x', labelsize=10, pad=1)
cax.xaxis.label.set_size(10)

fig.savefig(FIG_DIR / 'child_adult_rsa_summary_figure.png', dpi=300, bbox_inches='tight')


## 9. Static AoA-centrality correlations

In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

OUTPUT_DIR = Path('outputs')
DATA_DIR = OUTPUT_DIR / 'analysis_data'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIG_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

MAIN_CUE_MIN_NVALID = 20
ASSOCIATION_MIN_REPORT_COUNT = 2
NETWORK_ORDER = ['child', 'adult', 'speech', 'book']
SOURCE_LABELS = {'child': 'Child associations', 'adult': 'Adult associations', 'speech': 'Child-directed speech', 'book': 'Child-directed book'}
SOURCE_COLORS = {
    'child': '#E1B229',
    'adult': '#2E6A87',
    'speech': '#5F8D78',
    'book': '#A9794B',
}

NODE_METRICS_OUT = DATA_DIR / 'network_node_metrics_wb_aoa_unweighted_raw_ge2_four_sources.csv'
WEIGHTED_NODE_METRICS_OUT = DATA_DIR / 'network_node_metrics_wb_aoa_weighted_raw_ge2_four_sources.csv'
WIDE_MAIN_OUT = DATA_DIR / 'aoa_wide_wb_unweighted_raw_ge2_nvalid20.csv'
WIDE_WEIGHTED_OUT = DATA_DIR / 'aoa_wide_wb_weighted_raw_ge2_nvalid20.csv'

relation_strengths = pd.read_csv(RELATION_PATH, encoding='utf-8-sig')
relation_strengths['cue'] = relation_strengths['cue'].astype(str).str.strip()
relation_strengths['response'] = relation_strengths['response'].astype(str).str.strip()
relation_strengths['source'] = relation_strengths['source'].astype(str).str.strip()

child_edges_full = relation_strengths.loc[relation_strengths['source'].eq('child')].copy()
child_edges_full = child_edges_full.rename(columns={'weight': 'child_weight', 'n_producers': 'n_children', 'n_valid_source': 'n_valid'})
child_edges_full['child_weight'] = pd.to_numeric(child_edges_full['child_weight'], errors='coerce')

adult_edges_full = relation_strengths.loc[relation_strengths['source'].eq('adult')].copy()
adult_edges_full = adult_edges_full.rename(columns={'weight': 'adult_weight', 'n_valid_source': 'n_valid_retained'})
adult_edges_full['adult_weight'] = pd.to_numeric(adult_edges_full['adult_weight'], errors='coerce')

env_edges_full = relation_strengths.loc[relation_strengths['source'].isin(['speech', 'book'])].copy()
env_edges_full = env_edges_full.rename(columns={'weight': 'normalized_weight'})
env_edges_full['normalized_weight'] = pd.to_numeric(env_edges_full['normalized_weight'], errors='coerce')

word_measures_local = pd.read_csv(WORD_MEASURES_PATH, encoding='utf-8-sig')
word_measures_local['word'] = word_measures_local['word'].astype(str).str.strip()
speech_freq = word_measures_local[['word', 'speech_freq', 'speech_freq_per_million']].rename(columns={'speech_freq_per_million': 'log_speech_freq_per_million'})
speech_freq['log_speech_freq_per_million'] = np.log1p(pd.to_numeric(word_measures_local['speech_freq_per_million'], errors='coerce'))
book_freq = word_measures_local[['word', 'book_freq', 'book_freq_per_million']].rename(columns={'book_freq_per_million': 'log_book_freq_per_million'})
book_freq['log_book_freq_per_million'] = np.log1p(pd.to_numeric(word_measures_local['book_freq_per_million'], errors='coerce'))

child_cue_summary = pd.read_csv(TABLE_DIR / 'child_cue_summary_for_association_validation.csv', encoding='utf-8-sig')
child_cue_summary['cue'] = child_cue_summary['cue'].astype(str).str.strip()
child_cue_summary['n_valid'] = pd.to_numeric(child_cue_summary['n_valid'], errors='coerce')

cue_meta = pd.read_csv(OUTPUT_SUMMARY, encoding='utf-8-sig')
cue_meta['cue'] = cue_meta['cue'].astype(str).str.strip()
cue_meta = cue_meta.drop_duplicates('cue', keep='first')

wb = word_measures_local[['word', 'wordbank_aoa']].dropna(subset=['wordbank_aoa']).copy()
wb = wb.rename(columns={'word': 'Word', 'wordbank_aoa': 'AoA'})
wb['AoA'] = pd.to_numeric(wb['AoA'], errors='coerce')
wb = wb[['Word', 'AoA']].drop_duplicates('Word', keep='first')

def canonical_pos_group_from_tag(tag):
    tag = (tag or '').strip().upper()
    if tag in {'NN', 'NR', 'NT'}:
        return 'noun'
    if tag in {'VV', 'VC', 'VE'}:
        return 'verb'
    if tag in {'JJ', 'VA', 'AD'}:
        return 'adj_adv'
    return 'other'

def canonical_pos_group(row):
    corrected = str(row.get('cue_class_4way_corrected_en', '')).strip().lower()
    if corrected == 'noun':
        return 'noun'
    if corrected == 'verb':
        return 'verb'
    if corrected in {'adj/adv', 'adjective/adverb', 'adj_adv'}:
        return 'adj_adv'
    return canonical_pos_group_from_tag(row.get('cue_POS', ''))

def zscore(s):
    s = pd.to_numeric(s, errors='coerce')
    std = s.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / std

cue_meta['pos_group'] = cue_meta.apply(canonical_pos_group, axis=1)

main_analysis_cues = set(child_cue_summary.loc[child_cue_summary['n_valid'] >= MAIN_CUE_MIN_NVALID, 'cue'])
main_analysis_cues &= set(adult_edges_full['cue'])
network_nodes = sorted(main_analysis_cues)

def build_directed_cue_network(edges: pd.DataFrame,
                               weight_col: str,
                               nodes: list[str],
                               min_count_col: str | None = None,
                               min_count: int | None = None) -> nx.DiGraph:
    node_set = set(nodes)
    sub = edges.loc[
        edges['cue'].isin(node_set)
        & edges['response'].isin(node_set)
        & (edges['cue'] != edges['response']),
        ['cue', 'response', weight_col] + ([min_count_col] if min_count_col else [])
    ].copy()
    sub[weight_col] = pd.to_numeric(sub[weight_col], errors='coerce')
    sub = sub.dropna(subset=[weight_col])
    sub = sub.loc[sub[weight_col] > 0]
    if min_count_col is not None and min_count is not None:
        sub[min_count_col] = pd.to_numeric(sub[min_count_col], errors='coerce')
        sub = sub.loc[sub[min_count_col] >= min_count]
    sub = sub.groupby(['cue', 'response'], as_index=False)[weight_col].sum()
    graph = nx.DiGraph()
    graph.add_nodes_from(nodes)
    for row in sub.itertuples(index=False):
        graph.add_edge(row.cue, row.response, weight=float(getattr(row, weight_col)))
    return graph

def directed_to_weighted_undirected(graph: nx.DiGraph) -> nx.Graph:
    undirected = nx.Graph()
    undirected.add_nodes_from(graph.nodes())
    for u, v, data in graph.edges(data=True):
        weight = float(data.get('weight', 1.0))
        if undirected.has_edge(u, v):
            undirected[u][v]['weight'] += weight
        else:
            undirected.add_edge(u, v, weight=weight)
    return undirected

def build_environment_undirected_network(edges: pd.DataFrame, source: str, nodes: list[str]) -> nx.Graph:
    node_set = set(nodes)
    sub = edges.loc[
        (edges['source'] == source)
        & edges['cue'].isin(node_set)
        & edges['response'].isin(node_set)
        & (edges['cue'] != edges['response']),
        ['cue', 'response', 'normalized_weight']
    ].copy()
    sub = sub.dropna(subset=['normalized_weight'])
    sub = sub.loc[sub['normalized_weight'] > 0]
    graph = nx.Graph()
    graph.add_nodes_from(nodes)
    for row in sub.itertuples(index=False):
        u, v = row.cue, row.response
        weight = float(row.normalized_weight)
        if graph.has_edge(u, v):
            graph[u][v]['weight'] += weight
        else:
            graph.add_edge(u, v, weight=weight)
    return graph

def directed_node_metrics(graph: nx.DiGraph, source: str, weighted: bool = False) -> pd.DataFrame:
    metric_name = 'weighted_in_degree' if weighted else 'in_degree'
    metric_values = [
        float(graph.in_degree(node, weight='weight')) if weighted else int(graph.in_degree(node))
        for node in graph.nodes()
    ]
    return pd.DataFrame({
        'cue': list(graph.nodes()),
        'source': source,
        'metric_name': metric_name,
        'metric_value': metric_values,
    })

def undirected_node_metrics(graph: nx.Graph, source: str, weighted: bool = False) -> pd.DataFrame:
    metric_name = 'weighted_degree' if weighted else 'degree'
    metric_values = [
        float(graph.degree(node, weight='weight')) if weighted else int(graph.degree(node))
        for node in graph.nodes()
    ]
    return pd.DataFrame({
        'cue': list(graph.nodes()),
        'source': source,
        'metric_name': metric_name,
        'metric_value': metric_values,
    })

child_directed = build_directed_cue_network(
    child_edges_full,
    'child_weight',
    network_nodes,
    min_count_col='n_children',
    min_count=ASSOCIATION_MIN_REPORT_COUNT,
)
adult_directed = build_directed_cue_network(
    adult_edges_full,
    'adult_weight',
    network_nodes,
    min_count_col='n_producers',
    min_count=ASSOCIATION_MIN_REPORT_COUNT,
)
speech_undirected = build_environment_undirected_network(env_edges_full, 'speech', network_nodes)
book_undirected = build_environment_undirected_network(env_edges_full, 'book', network_nodes)

network_density_summary = pd.DataFrame([
    {'source': 'child', 'network_type': 'directed_raw', 'n_nodes': len(network_nodes), 'n_directed_edges': child_directed.number_of_edges(), 'n_undirected_edges': directed_to_weighted_undirected(child_directed).number_of_edges(), 'undirected_density': nx.density(directed_to_weighted_undirected(child_directed))},
    {'source': 'adult', 'network_type': 'directed_raw', 'n_nodes': len(network_nodes), 'n_directed_edges': adult_directed.number_of_edges(), 'n_undirected_edges': directed_to_weighted_undirected(adult_directed).number_of_edges(), 'undirected_density': nx.density(directed_to_weighted_undirected(adult_directed))},
    {'source': 'speech', 'network_type': 'undirected_raw_ppmi', 'n_nodes': len(network_nodes), 'n_directed_edges': np.nan, 'n_undirected_edges': speech_undirected.number_of_edges(), 'undirected_density': nx.density(speech_undirected)},
    {'source': 'book', 'network_type': 'undirected_raw_ppmi', 'n_nodes': len(network_nodes), 'n_directed_edges': np.nan, 'n_undirected_edges': book_undirected.number_of_edges(), 'undirected_density': nx.density(book_undirected)},
])
network_density_summary.to_csv(TABLE_DIR / 'aoa_network_density_summary_raw_unweighted_ge2_nvalid20.csv', index=False)

node_metrics = pd.concat([
    directed_node_metrics(child_directed, 'child'),
    directed_node_metrics(adult_directed, 'adult'),
    undirected_node_metrics(speech_undirected, 'speech'),
    undirected_node_metrics(book_undirected, 'book'),
], ignore_index=True)
node_metrics.to_csv(NODE_METRICS_OUT, index=False)

weighted_node_metrics = pd.concat([
    directed_node_metrics(child_directed, 'child', weighted=True),
    directed_node_metrics(adult_directed, 'adult', weighted=True),
    undirected_node_metrics(speech_undirected, 'speech', weighted=True),
    undirected_node_metrics(book_undirected, 'book', weighted=True),
], ignore_index=True)
weighted_node_metrics.to_csv(WEIGHTED_NODE_METRICS_OUT, index=False)

def prepare_wide(metrics_df: pd.DataFrame) -> pd.DataFrame:
    wide = (
        metrics_df.pivot_table(index='cue', columns='source', values='metric_value', aggfunc='first')
        .reindex(columns=NETWORK_ORDER)
        .reset_index()
    )
    for src in NETWORK_ORDER:
        wide[f'z_li_{src}'] = zscore(np.log1p(wide[src]))
    wide = wide.merge(cue_meta[['cue', 'cue_POS', 'pos_group']], on='cue', how='left')
    wide = wide.merge(speech_freq.rename(columns={'word': 'cue'}), on='cue', how='left')
    wide = wide.merge(book_freq.rename(columns={'word': 'cue'}), on='cue', how='left')
    wide = wide.merge(wb.rename(columns={'Word': 'cue', 'AoA': 'aoa_wb'}), on='cue', how='left')
    wide['z_log_speech_freq_pm'] = zscore(wide['log_speech_freq_per_million'])
    wide['z_log_book_freq_pm'] = zscore(wide['log_book_freq_per_million'])
    wide = wide.drop_duplicates('cue', keep='first')
    return wide.loc[wide['pos_group'].isin(['noun', 'verb', 'adj_adv'])].copy()

wide_wb = prepare_wide(node_metrics)
wide_weighted_wb = prepare_wide(weighted_node_metrics)
wide_wb.to_csv(WIDE_MAIN_OUT, index=False)
wide_weighted_wb.to_csv(WIDE_WEIGHTED_OUT, index=False)

print(f'Main node set: {len(network_nodes):,} cues')
print(network_density_summary.round(4).to_string(index=False))
print(f'Rows after POS filter: {len(wide_wb):,}')
print(f'Rows with WB AoA and complete unweighted predictors: {wide_wb.dropna(subset=["aoa_wb"] + [f"z_li_{n}" for n in NETWORK_ORDER]).shape[0]:,}')
print(f'Rows with WB AoA and complete weighted predictors: {wide_weighted_wb.dropna(subset=["aoa_wb"] + [f"z_li_{n}" for n in NETWORK_ORDER]).shape[0]:,}')


In [ ]:
def p_to_apa(p_value: float) -> str:
    if pd.isna(p_value):
        return '= NA'
    if p_value < 0.001:
        return '< .001'
    return f'= {p_value:.3f}'.replace('0.', '.')

def sourcewise_aoa_correlations(df: pd.DataFrame, metric_family: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for net in NETWORK_ORDER:
        pred = f'z_li_{net}'
        sub = df[['cue', 'aoa_wb', pred]].dropna().copy()
        if len(sub) < 3:
            pearson_r = pearson_p = spearman_rho = spearman_p = np.nan
        else:
            pearson_r, pearson_p = stats.pearsonr(sub[pred], sub['aoa_wb'])
            spearman_rho, spearman_p = stats.spearmanr(sub[pred], sub['aoa_wb'])
        rows.append({
            'metric_family': metric_family,
            'source': net,
            'source_label': SOURCE_LABELS[net],
            'n': len(sub),
            'metric': ('weighted_' if metric_family == 'weighted' else 'unweighted_') + ('in_degree' if net in {'child', 'adult'} else 'degree'),
            'pearson_r': pearson_r,
            'pearson_p': pearson_p,
            'spearman_rho': spearman_rho,
            'spearman_p': spearman_p,
        })
    corr = pd.DataFrame(rows)
    apa = corr.assign(
        pearson=lambda x: x.apply(lambda r: f"r = {r.pearson_r:.2f}, p {p_to_apa(r.pearson_p)}", axis=1),
        spearman=lambda x: x.apply(lambda r: f"rho = {r.spearman_rho:.2f}, p {p_to_apa(r.spearman_p)}", axis=1),
    )[[
        'metric_family', 'source_label', 'n', 'metric', 'pearson', 'spearman'
    ]].rename(columns={
        'metric_family': 'Metric family',
        'source_label': 'Source',
        'n': 'N',
        'metric': 'Network metric',
        'pearson': 'Pearson correlation',
        'spearman': 'Spearman correlation',
    })
    return corr, apa

wb_aoa_static_correlations, wb_aoa_static_correlations_apa = sourcewise_aoa_correlations(wide_wb, 'unweighted')
wb_aoa_weighted_static_correlations, wb_aoa_weighted_static_correlations_apa = sourcewise_aoa_correlations(wide_weighted_wb, 'weighted')
wb_aoa_all_static_correlations = pd.concat([
    wb_aoa_static_correlations,
    wb_aoa_weighted_static_correlations,
], ignore_index=True)
wb_aoa_all_static_correlations_apa = pd.concat([
    wb_aoa_static_correlations_apa,
    wb_aoa_weighted_static_correlations_apa,
], ignore_index=True)

wb_aoa_static_correlations.to_csv(TABLE_DIR / 'wb_aoa_unweighted_static_correlations_ge2_nvalid20.csv', index=False)
wb_aoa_static_correlations_apa.to_csv(TABLE_DIR / 'wb_aoa_unweighted_static_correlations_APA_ge2_nvalid20.csv', index=False)
wb_aoa_weighted_static_correlations.to_csv(TABLE_DIR / 'wb_aoa_weighted_static_correlations_ge2_nvalid20.csv', index=False)
wb_aoa_weighted_static_correlations_apa.to_csv(TABLE_DIR / 'wb_aoa_weighted_static_correlations_APA_ge2_nvalid20.csv', index=False)
wb_aoa_all_static_correlations.to_csv(TABLE_DIR / 'wb_aoa_unweighted_weighted_static_correlations_ge2_nvalid20.csv', index=False)
wb_aoa_all_static_correlations_apa.to_csv(TABLE_DIR / 'wb_aoa_unweighted_weighted_static_correlations_APA_ge2_nvalid20.csv', index=False)

print('WB AoA static source-wise correlations:')
print(wb_aoa_all_static_correlations.round(4).to_string(index=False))
print('\nAPA-formatted correlations:')
print(wb_aoa_all_static_correlations_apa.to_string(index=False))


In [ ]:
def make_static_aoa_plot_long(df: pd.DataFrame, metric_family: str) -> pd.DataFrame:
    plot_parts = []
    for net in NETWORK_ORDER:
        cols = ['cue', 'aoa_wb', f'z_li_{net}']
        sub = df[cols].dropna().copy()
        sub = sub.rename(columns={f'z_li_{net}': 'standardized_degree'})
        sub['source'] = net
        sub['source_label'] = SOURCE_LABELS[net]
        sub['metric_family'] = metric_family
        plot_parts.append(sub)
    return pd.concat(plot_parts, ignore_index=True)

plot_long = pd.concat([
    make_static_aoa_plot_long(wide_wb, 'Unweighted'),
    make_static_aoa_plot_long(wide_weighted_wb, 'Weighted'),
], ignore_index=True)

corr_lookup = {
    (row.metric_family.capitalize(), row.source): (row.spearman_rho, row.spearman_p)
    for row in wb_aoa_all_static_correlations.itertuples(index=False)
}

def stars_from_p(p_value: float) -> str:
    if pd.isna(p_value):
        return ''
    if p_value < 0.001:
        return '***'
    if p_value < 0.01:
        return '**'
    if p_value < 0.05:
        return '*'
    return ''

fig, axes = plt.subplots(4, 2, figsize=(7.2, 10.2), sharex=True, sharey=False)
for row_idx, net in enumerate(NETWORK_ORDER):
    for col_idx, metric_family in enumerate(['Unweighted', 'Weighted']):
        ax = axes[row_idx, col_idx]
        sub = plot_long.loc[
            (plot_long['metric_family'] == metric_family)
            & (plot_long['source'] == net)
        ]
        sns.regplot(
            data=sub,
            x='aoa_wb',
            y='standardized_degree',
            scatter=True,
            ci=95,
            truncate=False,
            color=SOURCE_COLORS[net],
            scatter_kws={'s': 17, 'alpha': 0.34, 'edgecolor': 'none'},
            line_kws={'lw': 2.6},
            ax=ax,
        )
        rho, rho_p = corr_lookup.get((metric_family, net), (np.nan, np.nan))
        ax.text(
            0.95,
            0.92,
            f'rho = {rho:.2f}{stars_from_p(rho_p)}',
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=10.5,
            fontweight='bold',
            color=SOURCE_COLORS[net],
        )
        if row_idx == 0:
            ax.set_title(metric_family, fontweight='bold')
        metric_label = 'standardized in-degree' if net in {'child', 'adult'} else 'standardized degree'
        if col_idx == 0:
            ax.set_ylabel(f'{SOURCE_LABELS[net]}\n{metric_label}', color='black', fontweight='bold')
        else:
            ax.set_ylabel('')
        if row_idx == len(NETWORK_ORDER) - 1:
            ax.set_xlabel('Wordbank AoA')
        else:
            ax.set_xlabel('')
        ax.grid(False)
        sns.despine(ax=ax)

fig.tight_layout(w_pad=1.1, h_pad=0.78)
fig.savefig(FIG_DIR / 'wb_aoa_unweighted_weighted_degree_scatter.png', dpi=300, bbox_inches='tight')


## 10. Lexical growth models

Growth values are computed month by month for preferential attachment, preferential acquisition, and lure of associates.

In [ ]:
from scipy import optimize
from statsmodels.stats.multitest import multipletests


MODEL_ORDER = [
    "preferential_attachment",
    "preferential_acquisition",
    "lure_of_associates",
]
MODEL_LABELS = {
    "preferential_attachment": "Preferential attachment",
    "preferential_acquisition": "Preferential acquisition",
    "lure_of_associates": "Lure of associates",
}


def zscore_series(values: pd.Series) -> pd.Series:
    values = pd.to_numeric(pd.Series(values), errors="coerce")
    sd = values.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - values.mean()) / sd


def p_to_stars(p_value: float) -> str:
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return ""


def _build_directed_association(
    edges: pd.DataFrame,
    nodes: list[str],
    weight_col: str,
    count_col: str,
    min_count: int | None,
) -> nx.DiGraph:
    node_set = set(nodes)
    sub = edges.loc[
        edges["cue"].isin(node_set)
        & edges["response"].isin(node_set)
        & (edges["cue"] != edges["response"])
    ].copy()
    sub[weight_col] = pd.to_numeric(sub[weight_col], errors="coerce")
    sub = sub.loc[sub[weight_col] > 0]
    if min_count is not None:
        sub[count_col] = pd.to_numeric(sub[count_col], errors="coerce")
        sub = sub.loc[sub[count_col] >= min_count]
    sub = sub.groupby(["cue", "response"], as_index=False)[weight_col].sum()

    graph = nx.DiGraph()
    graph.add_nodes_from(nodes)
    for row in sub.itertuples(index=False):
        graph.add_edge(row.cue, row.response, weight=float(getattr(row, weight_col)))
    return graph


def _build_directed_environment(edges: pd.DataFrame, source: str, nodes: list[str]) -> nx.DiGraph:
    node_set = set(nodes)
    sub = edges.loc[
        (edges["source"] == source)
        & edges["cue"].isin(node_set)
        & edges["response"].isin(node_set)
        & (edges["cue"] != edges["response"])
    ].copy()
    sub["normalized_weight"] = pd.to_numeric(sub["normalized_weight"], errors="coerce")
    sub = sub.loc[sub["normalized_weight"] > 0]

    graph = nx.DiGraph()
    graph.add_nodes_from(nodes)
    for row in sub.itertuples(index=False):
        if graph.has_edge(row.cue, row.response):
            graph[row.cue][row.response]["weight"] += float(row.normalized_weight)
        else:
            graph.add_edge(row.cue, row.response, weight=float(row.normalized_weight))
    return graph


def _growth_values_for_month(
    graph: nx.DiGraph,
    aoa_map: dict[str, int],
    month: int,
) -> pd.DataFrame:
    known = {word for word, aoa in aoa_map.items() if aoa <= month}
    candidates = sorted(word for word, aoa in aoa_map.items() if aoa > month)
    learned_next = {word for word, aoa in aoa_map.items() if aoa == month + 1}
    if len(candidates) < 2 or not learned_next:
        return pd.DataFrame()
    known_subgraph = graph.subgraph(known)

    rows = []
    for word in candidates:
        known_targets = [target for _, target in graph.out_edges(word) if target in known]
        pa = (
            float(np.mean([known_subgraph.in_degree(target) for target in known_targets]))
            if known_targets
            else 0.0
        )
        pacq = float(graph.in_degree(word))
        loa = float(sum(1 for known_word, _ in graph.in_edges(word) if known_word in known))
        rows.extend(
            [
                {
                    "growth_model": "preferential_attachment",
                    "word": word,
                    "growth_raw": pa,
                },
                {
                    "growth_model": "preferential_acquisition",
                    "word": word,
                    "growth_raw": pacq,
                },
                {
                    "growth_model": "lure_of_associates",
                    "word": word,
                    "growth_raw": loa,
                },
            ]
        )

    values = pd.DataFrame(rows)
    if values.empty:
        return values
    values["month"] = month
    values["is_learned_next"] = values["word"].isin(learned_next)
    values["n_known"] = len(known)
    values["n_candidates"] = len(candidates)
    values["n_learned_next"] = len(learned_next)
    values["growth_log"] = np.log1p(values["growth_raw"])
    values["growth_z_within_month"] = values.groupby("growth_model")["growth_log"].transform(zscore_series)
    return values


def _softmax_summary(values: pd.DataFrame, n_words_total: int) -> pd.DataFrame:
    rows = []
    for (source, growth_model), sub in values.groupby(["source", "growth_model"]):
        month_payload = []
        loglik_null = 0.0
        n_observations = 0
        for _, month_df in sub.groupby("month"):
            month_df = month_df.sort_values("word").reset_index(drop=True)
            learned_idx = month_df.index[month_df["is_learned_next"]].to_numpy()
            if len(month_df) < 2 or len(learned_idx) == 0:
                continue
            x = month_df["growth_z_within_month"].to_numpy(dtype=float)
            month_payload.append((x, learned_idx))
            loglik_null += len(learned_idx) * (-np.log(len(month_df)))
            n_observations += len(learned_idx)

        def neg_loglik(beta_array):
            beta = float(beta_array[0])
            loglik = 0.0
            for x, learned_idx in month_payload:
                eta = beta * x
                max_eta = np.max(eta)
                log_denom = max_eta + np.log(np.exp(eta - max_eta).sum())
                loglik += float(np.sum(eta[learned_idx] - log_denom))
            return -loglik

        fit = optimize.minimize(neg_loglik, x0=np.array([0.0]), method="BFGS")
        beta = float(fit.x[0])
        loglik_model = -float(fit.fun)
        chi2 = max(0.0, 2 * (loglik_model - loglik_null))
        p_value = float(stats.chi2.sf(chi2, 1))
        rows.append(
            {
                "source": source,
                "growth_model": growth_model,
                "n_observations": n_observations,
                "n_words_total": n_words_total,
                "beta": beta,
                "loglik_null": loglik_null,
                "loglik_model": loglik_model,
                "chi2": chi2,
                "p": p_value,
                "p_fdr": np.nan,
                "bic_improvement": chi2 - np.log(n_observations),
                "converged": bool(fit.success),
            }
        )

    summary = pd.DataFrame(rows)
    summary["p_fdr"] = multipletests(summary["p"], method="fdr_bh")[1]
    return summary


def run_wb_growth_models(
    base_dir: Path,
    source_labels: dict[str, str],
    main_cue_min_nvalid: int = 20,
    min_report_count: int = 2,
) -> dict[str, pd.DataFrame]:
    input_dir = base_dir / "data"
    output_dir = base_dir / "outputs"
    data_dir = output_dir / "analysis_data"
    table_dir = output_dir / "tables"

    relations = pd.read_csv(input_dir / "analysis_relation_strengths.csv", encoding="utf-8-sig")
    cue_summary = pd.read_csv(table_dir / "child_cue_summary_for_association_validation.csv", encoding="utf-8-sig")
    word_measures = pd.read_csv(input_dir / "analysis_word_measures.csv", encoding="utf-8-sig")

    relations["cue"] = relations["cue"].astype(str).str.strip()
    relations["response"] = relations["response"].astype(str).str.strip()
    relations["source"] = relations["source"].astype(str).str.strip()
    child = relations.loc[relations["source"].eq("child")].rename(columns={"weight": "child_weight", "n_producers": "n_children"}).copy()
    adult = relations.loc[relations["source"].eq("adult")].rename(columns={"weight": "raw_weight", "n_producers": "n_producers"}).copy()
    environment = relations.loc[relations["source"].isin(["speech", "book"])].rename(columns={"weight": "normalized_weight"}).copy()
    cue_summary["cue"] = cue_summary["cue"].astype(str).str.strip()
    cue_summary["n_valid"] = pd.to_numeric(cue_summary["n_valid"], errors="coerce")
    word_measures["word"] = word_measures["word"].astype(str).str.strip()
    word_measures["wordbank_aoa"] = pd.to_numeric(word_measures["wordbank_aoa"], errors="coerce")

    network_nodes = sorted(
        set(cue_summary.loc[cue_summary["n_valid"] >= main_cue_min_nvalid, "cue"])
        & set(adult["cue"])
    )
    aoa = (
        word_measures[["word", "wordbank_aoa"]]
        .dropna(subset=["word", "wordbank_aoa"])
        .drop_duplicates("word", keep="first")
        .rename(columns={"wordbank_aoa": "aoa_month"})
    )
    aoa["prop_at_aoa"] = np.nan
    analysis_nodes = sorted(set(network_nodes) & set(aoa["word"]))
    aoa = aoa.loc[aoa["word"].isin(analysis_nodes)].copy()
    aoa_map = dict(zip(aoa["word"], aoa["aoa_month"]))

    graphs = {
        "child": _build_directed_association(
            child,
            analysis_nodes,
            weight_col="child_weight",
            count_col="n_children",
            min_count=min_report_count,
        ),
        "adult": _build_directed_association(
            adult,
            analysis_nodes,
            weight_col="raw_weight",
            count_col="n_producers",
            min_count=min_report_count,
        ),
        "speech": _build_directed_environment(environment, "speech", analysis_nodes),
        "book": _build_directed_environment(environment, "book", analysis_nodes),
    }

    density_rows = [
        {
            "source": source,
            "source_label": source_labels[source],
            "n_nodes": len(analysis_nodes),
            "n_directed_edges": graph.number_of_edges(),
            "directed_density": graph.number_of_edges() / (len(analysis_nodes) * (len(analysis_nodes) - 1)),
            "min_report_count": min_report_count if source in {"child", "adult"} else np.nan,
        }
        for source, graph in graphs.items()
    ]

    value_rows = []
    for source, graph in graphs.items():
        for month in range(16, 30):
            month_values = _growth_values_for_month(graph, aoa_map, month)
            if not month_values.empty:
                month_values["source"] = source
                month_values["source_label"] = source_labels[source]
                month_values["growth_model_label"] = month_values["growth_model"].map(MODEL_LABELS)
                value_rows.append(month_values)
    values = pd.concat(value_rows, ignore_index=True)
    learned = values.loc[values["is_learned_next"]].copy()

    mean_rows = []
    for (source, growth_model), sub in learned.groupby(["source", "growth_model"]):
        t_stat, p_value = stats.ttest_1samp(sub["growth_z_within_month"], 0, nan_policy="omit")
        mean_rows.append(
            {
                "source": source,
                "source_label": source_labels[source],
                "growth_model": growth_model,
                "growth_model_label": MODEL_LABELS[growth_model],
                "n_observations": len(sub),
                "n_words": sub["word"].nunique(),
                "mean_z": sub["growth_z_within_month"].mean(),
                "se_z": sub["growth_z_within_month"].sem(),
                "sd_z": sub["growth_z_within_month"].std(ddof=1),
                "t": t_stat,
                "df": len(sub) - 1,
                "p": p_value,
                "p_fdr": np.nan,
                "stars": p_to_stars(p_value),
            }
        )
    mean_summary = pd.DataFrame(mean_rows)
    mean_summary["p_fdr"] = multipletests(mean_summary["p"], method="fdr_bh")[1]

    softmax_summary = _softmax_summary(values, n_words_total=len(aoa))
    softmax_summary = softmax_summary.merge(
        mean_summary[["source", "source_label", "growth_model", "growth_model_label"]],
        on=["source", "growth_model"],
        how="left",
    )

    outputs = {
        "aoa": aoa,
        "network_density": pd.DataFrame(density_rows),
        "values": values,
        "learned_observations": learned,
        "mean_summary": mean_summary,
        "softmax_summary": softmax_summary,
    }
    return outputs


In [ ]:
growth_outputs = run_wb_growth_models(
    Path('.'),
    source_labels=SOURCE_LABELS,
    main_cue_min_nvalid=MAIN_CUE_MIN_NVALID,
    min_report_count=ASSOCIATION_MIN_REPORT_COUNT,
)

growth_outputs['aoa'].to_csv(TABLE_DIR / 'wb_growth_three_models_aoa_ge2_main.csv', index=False)
growth_outputs['network_density'].to_csv(TABLE_DIR / 'wb_growth_three_models_network_density_ge2_main.csv', index=False)
growth_outputs['values'].to_csv(TABLE_DIR / 'wb_growth_three_models_values_ge2_main.csv', index=False)
growth_outputs['learned_observations'].to_csv(TABLE_DIR / 'wb_growth_three_models_learned_observations_ge2_main.csv', index=False)
growth_outputs['mean_summary'].to_csv(TABLE_DIR / 'wb_growth_three_models_mean_z_summary_ge2_main.csv', index=False)

print(f"Growth-model words: {len(growth_outputs['aoa']):,}")
print('\nNetwork density:')
print(growth_outputs['network_density'].round(4).to_string(index=False))
print('\nMean z summary:')
print(growth_outputs['mean_summary'].round(4).to_string(index=False))


In [ ]:
growth_plot = growth_outputs['mean_summary'].copy()
growth_plot['growth_model'] = pd.Categorical(growth_plot['growth_model'], MODEL_ORDER, ordered=True)
growth_plot['source'] = pd.Categorical(growth_plot['source'], NETWORK_ORDER, ordered=True)
growth_plot = growth_plot.sort_values(['growth_model', 'source'])

model_labels = {
    'preferential_attachment': 'Preferential\nattachment',
    'preferential_acquisition': 'Preferential\nacquisition',
    'lure_of_associates': 'Lure of\nassociates',
}
x = np.arange(len(MODEL_ORDER))
bar_width = 0.18
offsets = np.linspace(-1.5 * bar_width, 1.5 * bar_width, len(NETWORK_ORDER))

fig, ax = plt.subplots(figsize=(9.2, 5.8))
for offset, source in zip(offsets, NETWORK_ORDER):
    sub = growth_plot.loc[growth_plot['source'] == source].set_index('growth_model').reindex(MODEL_ORDER)
    xpos = x + offset
    ax.bar(
        xpos,
        sub['mean_z'],
        width=bar_width,
        yerr=sub['se_z'],
        color=SOURCE_COLORS[source],
        edgecolor='white',
        linewidth=0.8,
        capsize=3,
        label=SOURCE_LABELS[source],
    )
    for px, row in zip(xpos, sub.itertuples()):
        if isinstance(row.stars, str) and row.stars:
            ax.text(
                px,
                row.mean_z + row.se_z + 0.012,
                row.stars,
                ha='center',
                va='bottom',
                fontsize=17,
                fontweight='bold',
                color=SOURCE_COLORS[source],
            )

ax.axhline(0, color='black', lw=1, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels([model_labels[m] for m in MODEL_ORDER], fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('Mean growth value of learned words', fontsize=15)
ax.tick_params(axis='y', labelsize=14)
ax.legend(frameon=False, title='Network', ncol=2, loc='upper center', bbox_to_anchor=(0.5, -0.18), fontsize=13, title_fontsize=14)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(FIG_DIR / 'wb_growth_three_models_mean_z_ge2_main.png', dpi=300, bbox_inches='tight')


In [ ]:

# Softmax model comparisons against a lexical-frequency baseline.
# Frequency is taken from the derived CCLOWW frequency column in the compact word-level table.

lemma_freq = pd.read_csv(WORD_MEASURES_PATH, encoding='utf-8-sig')
lemma_freq['word'] = lemma_freq['word'].astype(str).str.strip()
lemma_freq['Freq'] = pd.to_numeric(lemma_freq['ccloww_freq'], errors='coerce').fillna(0)
lemma_freq = lemma_freq[['word', 'Freq']].drop_duplicates('word', keep='first')

growth_values = growth_outputs['values'].merge(lemma_freq, on='word', how='left')
growth_values['freq_log'] = np.log1p(growth_values['Freq'].fillna(0))
growth_values['freq_z'] = zscore_series(growth_values['freq_log'])


def fit_softmax_choice(df: pd.DataFrame, predictors: list[str]) -> dict:
    month_payload = []
    n_observations = 0
    for _, month_df in df.groupby('month'):
        month_df = month_df.sort_values('word').reset_index(drop=True)
        learned_idx = month_df.index[month_df['is_learned_next']].to_numpy()
        if len(month_df) < 2 or len(learned_idx) == 0:
            continue
        X = month_df[predictors].to_numpy(dtype=float)
        month_payload.append((X, learned_idx))
        n_observations += len(learned_idx)

    def neg_loglik(beta):
        loglik = 0.0
        for X, learned_idx in month_payload:
            eta = X @ beta
            max_eta = np.max(eta)
            log_denom = max_eta + np.log(np.exp(eta - max_eta).sum())
            loglik += float(np.sum(eta[learned_idx] - log_denom))
        return -loglik

    fit = optimize.minimize(neg_loglik, x0=np.zeros(len(predictors)), method='BFGS')
    return {
        'beta': fit.x,
        'loglik': -float(fit.fun),
        'n_observations': n_observations,
        'converged': bool(fit.success),
    }


def softmax_against_frequency_baseline(values: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (source, growth_model), sub in values.groupby(['source', 'growth_model']):
        baseline = fit_softmax_choice(sub, ['freq_z'])
        full = fit_softmax_choice(sub, ['freq_z', 'growth_z_within_month'])
        chi2 = max(0.0, 2 * (full['loglik'] - baseline['loglik']))
        rows.append({
            'network': source,
            'network_label': SOURCE_LABELS[source],
            'growth_model': growth_model,
            'growth_model_label': MODEL_LABELS[growth_model],
            'n_observations': full['n_observations'],
            'df': 1,
            'loglik_baseline': baseline['loglik'],
            'loglik_full': full['loglik'],
            'delta_loglik': full['loglik'] - baseline['loglik'],
            'chi2': chi2,
            'p': stats.chi2.sf(chi2, 1),
            'p_fdr': np.nan,
            'aic_improvement': chi2 - 2,
            'bic_improvement': chi2 - np.log(full['n_observations']),
            'beta_frequency_full': full['beta'][0],
            'beta_growth': full['beta'][1],
            'converged_baseline': baseline['converged'],
            'converged_full': full['converged'],
        })
    out = pd.DataFrame(rows)
    out['p_fdr'] = multipletests(out['p'], method='fdr_bh')[1]
    out = out.sort_values(['growth_model', 'network']).reset_index(drop=True)
    return out


def drop_one_network_comparisons(values: pd.DataFrame) -> pd.DataFrame:
    specs = {
        'preferential_acquisition': ['child', 'adult', 'speech'],
        'lure_of_associates': ['child', 'speech', 'book'],
    }
    rows = []
    for growth_model, sources in specs.items():
        sub = values.loc[(values['growth_model'] == growth_model) & values['source'].isin(sources)].copy()
        wide = (
            sub.pivot_table(
                index=['month', 'word', 'is_learned_next', 'freq_z'],
                columns='source',
                values='growth_z_within_month',
                aggfunc='first',
            )
            .reset_index()
            .dropna(subset=['freq_z'] + sources)
        )
        full_predictors = ['freq_z'] + sources
        full = fit_softmax_choice(wide, full_predictors)
        for source in sources:
            reduced_sources = [s for s in sources if s != source]
            reduced = fit_softmax_choice(wide, ['freq_z'] + reduced_sources)
            chi2 = max(0.0, 2 * (full['loglik'] - reduced['loglik']))
            rows.append({
                'growth_model': growth_model,
                'growth_model_label': MODEL_LABELS[growth_model],
                'network_unique': source,
                'network_label': SOURCE_LABELS[source],
                'full_networks': ' + '.join(sources),
                'restricted_networks': ' + '.join(reduced_sources),
                'n_observations': full['n_observations'],
                'loglik_reduced': reduced['loglik'],
                'loglik_full': full['loglik'],
                'delta_loglik': full['loglik'] - reduced['loglik'],
                'chi2': chi2,
                'df': 1,
                'p': stats.chi2.sf(chi2, 1),
                'p_fdr': np.nan,
                'aic_improvement': chi2 - 2,
                'bic_improvement': chi2 - np.log(full['n_observations']),
                'beta_network_full': full['beta'][full_predictors.index(source)],
                'converged_reduced': reduced['converged'],
                'converged_full': full['converged'],
            })
    out = pd.DataFrame(rows)
    out['p_fdr'] = multipletests(out['p'], method='fdr_bh')[1]
    return out

baseline_growth_summary = softmax_against_frequency_baseline(growth_values)
unique_growth_summary = drop_one_network_comparisons(growth_values)

baseline_growth_summary.to_csv(TABLE_DIR / 'wb_growth_table2_against_frequency_baseline.csv', index=False)
unique_growth_summary.to_csv(TABLE_DIR / 'wb_growth_table3_unique_contributions.csv', index=False)

print('Growth models against frequency baseline:')
display(baseline_growth_summary.round(4))
print('Unique network contributions:')
display(unique_growth_summary.round(4))
